# ScamShield BD — Phase 3A Solution (Expanded Scenarios, Paraphrases & Cross-Scenario Check)
**Team Name:** Shield

## Declaration
- **Required Libraries & Versions:** `python==3.14.1`, `numpy==2.5.1`, `pandas==3.1.1`, `scipy==1.18.1`, `scikit-learn==1.9.1`, `python-dotenv==1.2.4`
- **External Datasets:** None. Training uses only `sample.csv` plus programmatically generated synthetic/noisy/paraphrased data.
- **Pretrained Models:** None.
- **APIs Used:** None (no API calls; `.env` is empty).
- **Data Generation & Augmentation:**
  - Expanded compositional generator (`data/synthetic_v2.csv`, `15,000` rows): 21 scenarios in Class 0, 16 in Class 1, 15 in Class 2, 15 in Class 3, 15 in Class 4, 20 in Class 5 across English/Bangla/Banglish/Mixed (~25% each), ~31.5% multi-turn, ~30.5% noisy (`apply_noise`), hard-negative contrast twins, plus `860` paraphrase variants (`10` per `sample.csv` row, `source='real_paraphrase'`).
  - No rows, labels, or predictions from `test.csv` are ever used in training or augmentation.
- **Methodology & Validation Strategy:**
  - *Preprocessing:* Unicode NFKC normalization; removal of `\u200b`, `\u200c`, `\ufeff` (preserving `\u200d` ZWJ for Bangla); Cyrillic/Greek-to-Latin homoglyph mapping; Latin-only lowercasing; `[URL]` and `[NUM]` tokens; `|||` preservation.
  - *Features & Models:* `char_wb` `(2, 5)` + `word` `(1, 2)` TF-IDF (`sublinear_tf=True`) + `LogisticRegression(class_weight='balanced')` for Task A & C, `OneVsRestClassifier(LogisticRegression(class_weight='balanced'))` for Task B.
  - *Validation:* Scenario-held-out `StratifiedGroupKFold`, real-row check (`synthetic_v2` only & 5-fold with train-fold paraphrases only), and cross-scenario cosine leakage check.
  - *Final Model:* Trained on `sample.csv` (`5x` weight) + `data/synthetic_v2.csv` (`1x` weight), outputting `submission_phase3a.csv`.
- **Random Seeds:** `SEED = 42` for `random`, `numpy`, and `sklearn`.
- **How to Reproduce:** Place `sample.csv`, `test.csv`, and `sample_submission.csv` in the working directory (or `./data/`) and run all cells top to bottom.


In [1]:
import os
import re
import time
import random
import unicodedata
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.preprocessing import normalize
from dotenv import load_dotenv

load_dotenv()

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = 'data' if os.path.isfile(os.path.join('data', 'sample.csv')) else '.'
TECH_CODES = ['T1', 'T2', 'T3', 'T4', 'T5', 'T6']
print('Environment ready. SEED =', SEED, '| DATA_DIR =', DATA_DIR)


Environment ready. SEED = 42 | DATA_DIR = .


## 1. Preprocessing

In [2]:
import re
import unicodedata

# Common Cyrillic / Greek homoglyphs mapped to Latin equivalents
HOMOGLYPH_MAP = {
    # Cyrillic lowercase
    "\u0430": "a",  # а -> a
    "\u0435": "e",  # е -> e
    "\u0451": "e",  # ё -> e
    "\u0456": "i",  # і -> i
    "\u0457": "i",  # ї -> i
    "\u0458": "j",  # ј -> j
    "\u043e": "o",  # о -> o
    "\u0440": "p",  # р -> p
    "\u0441": "c",  # с -> c
    "\u0443": "y",  # у -> y
    "\u0445": "x",  # х -> x
    "\u0455": "s",  # ѕ -> s
    "\u0501": "d",  # ԁ -> d
    "\u051b": "q",  # ԛ -> q
    "\u051d": "w",  # ԝ -> w
    # Cyrillic uppercase
    "\u0410": "A",  # А -> A
    "\u0412": "B",  # В -> B
    "\u0415": "E",  # Е -> E
    "\u041a": "K",  # К -> K
    "\u041c": "M",  # М -> M
    "\u041d": "H",  # Н -> H
    "\u041e": "O",  # О -> O
    "\u0420": "P",  # Р -> P
    "\u0421": "C",  # С -> C
    "\u0422": "T",  # Т -> T
    "\u0425": "X",  # Х -> X
    "\u04ae": "Y",  # Ү -> Y
    "\u0406": "I",  # І -> I
    # Greek lowercase / uppercase
    "\u03b1": "a",  # α -> a
    "\u03b5": "e",  # ε -> e
    "\u03b9": "i",  # ι -> i
    "\u03bf": "o",  # ο -> o
    "\u03c1": "p",  # ρ -> p
    "\u03c2": "c",  # ς -> c
    "\u0391": "A",  # Α -> A
    "\u0392": "B",  # Β -> B
    "\u0395": "E",  # Ε -> E
    "\u0397": "H",  # Η -> H
    "\u0399": "I",  # Ι -> I
    "\u039a": "K",  # Κ -> K
    "\u039c": "M",  # Μ -> M
    "\u039d": "N",  # Ν -> N
    "\u039f": "O",  # Ο -> O
    "\u03a1": "P",  # Ρ -> P
    "\u03a4": "T",  # Τ -> T
    "\u03a7": "X",  # Χ -> X
}
HOMOGLYPH_TRANS = str.maketrans(HOMOGLYPH_MAP)

# Defanged URL pattern: matches hxxp(s)://..., http(s)://..., or domain with [.]
URL_PATTERN = re.compile(
    r"(?:hxxps?|https?|ftp)://\S+|(?:[a-zA-Z0-9-]+\[\.\])+[a-zA-Z]{2,}(?:/\S*)?",
    re.IGNORECASE,
)

# Digit runs (ASCII and Bengali digits)
DIGIT_RUN_PATTERN = re.compile(r"[0-9\u09e6-\u09ef]+(?:[.,][0-9\u09e6-\u09ef]+)*")

# Latin uppercase letters only (preserves Bangla Unicode untouched)
LATIN_UPPER_PATTERN = re.compile(r"[A-Z]+")


def preprocess_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    # 1. Unicode NFKC normalization
    t = unicodedata.normalize("NFKC", text)

    # 2. Remove only \u200b, \u200c, \ufeff. Keep \u200d (ZWJ for Bangla).
    t = t.replace("\u200b", "").replace("\u200c", "").replace("\ufeff", "")

    # 3. Map common Cyrillic/Greek homoglyphs to Latin
    t = t.translate(HOMOGLYPH_TRANS)

    # 4. Replace defanged URLs with [URL] token
    t = URL_PATTERN.sub(" [URL] ", t)

    # 5. Lowercase Latin text only (leaves Bangla and [URL] handled cleanly)
    t = LATIN_UPPER_PATTERN.sub(lambda m: m.group(0).lower(), t)
    t = t.replace("[url]", "[URL]")

    # 6. Replace digit runs with [NUM] token
    t = DIGIT_RUN_PATTERN.sub(" [NUM] ", t)

    # 7. Preserve "|||" as a spaced separator token and collapse repeated whitespace
    t = t.replace("|||", " ||| ")
    t = re.sub(r"\s+", " ", t).strip()
    return t


def run_unit_tests():
    # Test 1: Homoglyphs (Cyrillic о, а, і, е, р, с)
    s1 = "Cоngratulаtіоns! Рay Tаkа 500 via bKаsh"
    out1 = preprocess_text(s1)
    assert "congratulations!" in out1, f"Failed homoglyph: {out1}"
    assert "pay taka [NUM] via bkash" in out1, f"Failed homoglyph+num: {out1}"

    # Test 2: Zero-width chars (\u200b, \u200c removed; \u200d kept!)
    s2 = "ব\u200cসে Trx\u200bID\ufeff র\u200d্যাংক"
    out2 = preprocess_text(s2)
    assert "\u200b" not in out2 and "\u200c" not in out2 and "\ufeff" not in out2, f"Failed ZW strip: {out2}"
    assert "\u200d" in out2, f"Failed to preserve ZWJ \\u200d: {out2}"
    assert "trxid" in out2, f"Failed Latin lowercase: {out2}"

    # Test 3: Defanged URLs and conversation separator |||
    s3 = "S: Click hxxps://gp-gift[.]xyz/claim ||| R: Okay 12345"
    out3 = preprocess_text(s3)
    assert "[URL]" in out3, f"Failed URL token: {out3}"
    assert "|||" in out3, f"Failed separator preservation: {out3}"
    assert "[NUM]" in out3, f"Failed NUM token: {out3}"

    # Test 4: Bangla text + Bengali digits
    s4 = "আপনার ৫০০ টাকা ক্যাশ আউট সফল হয়েছে।"
    out4 = preprocess_text(s4)
    assert "আপনার [NUM] টাকা ক্যাশ আউট সফল হয়েছে।" == out4, f"Failed Bangla: {out4}"

    print("All preprocess_text unit tests PASSED!")

run_unit_tests()


All preprocess_text unit tests PASSED!


## 2. Noise Augmenter

In [3]:
import re
import random

HOMOGLYPH_SWAPS = {
    "a": "\u0430",  # Cyrillic а
    "e": "\u0435",  # Cyrillic е
    "i": "\u0456",  # Cyrillic і
    "o": "\u043e",  # Cyrillic о
    "p": "\u0440",  # Cyrillic р
    "c": "\u0441",  # Cyrillic с
    "x": "\u0445",  # Cyrillic х
    "y": "\u0443",  # Cyrillic у
}

LEET_MAP = {
    "a": "4",
    "e": "3",
    "i": "1",
    "o": "0",
    "s": "5",
}

EMOJIS = ["📲", "🔥", "⚠️", "🎁", "💰", "✅", "🚨", "💸", "📦", "🎉"]
NOISE_STRINGS = ["wkrjfejejrn", "mnbvcx", "zzqwpo", "xkcdpl", "asdkjh", "qzxvbn", "lkjsdfoiu"]

BANGLISH_VARIANTS = [
    ("bhul", "vul"),
    ("korun", "koren"),
    ("bolen", "bolun"),
    ("pathan", "pathao"),
    ("taka", "tk"),
    ("apnar", "apnr"),
    ("ekhoni", "ekhn e"),
    ("taratari", "tara tari"),
    ("jitechen", "jitsen"),
    ("হয়েছে", "হয়েছে"),
    ("আয়", "আয়"),
    ("করুন", "করেন"),
    ("যোগ", "জোগ"),
]


def _corrupt_word(word: str, rng: random.Random, prob: float) -> str:
    if len(word) <= 2 or word in {"|||", "S:", "R:", "A:", "B:"} or word.startswith("http") or word.startswith("hxxp"):
        return word

    w = word
    # Bangla/Banglish variant
    if rng.random() < prob * 0.5:
        lw = w.lower()
        for src, dst in BANGLISH_VARIANTS:
            if lw == src:
                w = dst
                break
            elif lw == dst:
                w = src
                break

    # Zero-width char (\u200b or \u200c) inside word (works for both Latin & Bangla)
    if len(w) >= 4 and rng.random() < prob * 0.7:
        idx = rng.randint(1, len(w) - 1)
        zw = rng.choice(["\u200b", "\u200c"])
        w = w[:idx] + zw + w[idx:]

    # Latin-specific corruptions
    if re.search(r"[a-zA-Z]", w):
        chars = list(w)
        r = rng.random()
        if r < prob * 0.35 and len(chars) >= 4:
            # Swap two adjacent chars
            idx = rng.randint(1, len(chars) - 2)
            chars[idx], chars[idx + 1] = chars[idx + 1], chars[idx]
        elif r < prob * 0.6 and len(chars) >= 4:
            # Drop a char
            idx = rng.randint(1, len(chars) - 2)
            chars.pop(idx)
        elif r < prob * 0.85:
            # Repeat a char
            idx = rng.randint(0, len(chars) - 1)
            chars.insert(idx, chars[idx] * rng.randint(1, 2))
        w = "".join(chars)

        # Leet-speak or homoglyph swap
        if rng.random() < prob * 0.45:
            new_chars = []
            for ch in w:
                lch = ch.lower()
                if lch in HOMOGLYPH_SWAPS and rng.random() < 0.3:
                    new_chars.append(HOMOGLYPH_SWAPS[lch])
                elif lch in LEET_MAP and rng.random() < 0.25:
                    new_chars.append(LEET_MAP[lch])
                else:
                    new_chars.append(ch)
            w = "".join(new_chars)

        # Random capitalization
        if rng.random() < prob * 0.4:
            w = "".join(c.upper() if rng.random() < 0.4 else c.lower() for c in w)
    else:
        # Bangla script character drop / repeat / space split
        if len(w) >= 4 and rng.random() < prob * 0.35:
            idx = rng.randint(1, len(w) - 2)
            if rng.random() < 0.5:
                w = w[:idx] + " " + w[idx:]
            else:
                w = w[:idx] + w[idx + 1:]

    return w


def apply_noise(text: str, rng: random.Random, level: float = 0.3) -> str:
    """Apply realistic adversarial noise to English, Banglish, or Bangla text."""
    if not isinstance(text, str) or not text.strip():
        return text

    # Preserve ||| turns
    turns = text.split("|||")
    noisy_turns = []

    for turn in turns:
        tokens = turn.strip().split(" ")
        new_tokens = []
        for tok in tokens:
            # Defanged URL variation
            if "http" in tok.lower() or "[.]" in tok:
                if rng.random() < level:
                    tok = tok.replace("https://", "hxxps://").replace("http://", "hxxp://")
                    if "[.]" not in tok:
                        tok = tok.replace(".", "[.]")
                new_tokens.append(tok)
                continue

            if rng.random() < level:
                tok = _corrupt_word(tok, rng, prob=min(0.9, level + 0.2))
            new_tokens.append(tok)

            # Occasionally drop space between two tokens
            if len(new_tokens) >= 2 and rng.random() < level * 0.12:
                prev = new_tokens.pop()
                new_tokens[-1] = new_tokens[-1] + prev

        # Insert a random noise string
        if len(new_tokens) >= 3 and rng.random() < level * 0.4:
            pos = rng.randint(1, len(new_tokens) - 1)
            new_tokens.insert(pos, rng.choice(NOISE_STRINGS))

        # Insert an emoji
        if rng.random() < level * 0.5:
            pos = rng.choice([0, len(new_tokens)])
            new_tokens.insert(pos, rng.choice(EMOJIS))

        # Extra spaces occasionally
        joiner = "  " if rng.random() < level * 0.15 else " "
        noisy_turns.append(joiner.join(new_tokens))

    return " ||| ".join(noisy_turns)

## 3. Expanded Compositional Synthetic Generator (`synth_v2`)

In [4]:
import random
import pandas as pd

# Real + Invented Bangladeshi and generic brands/senders
MFS_BRANDS = [
    "bKash", "Nagad", "Rocket", "Upay", "SureCash", "Tap", "Cellfin",
    "PayBondhu", "TakaSwift", "DeshPay", "ChakaPay", "SonarWallet"
]
BANK_BRANDS = [
    "DBBL", "City Bank", "BRAC Bank", "Islami Bank", "Standard Chartered",
    "UCB", "EBL", "Mutual Trust Bank", "NodiBank", "ApexFin BD", "PadmaTrust Bank"
]
COURIER_BRANDS = [
    "Sundarban Courier", "SA Paribahan", "Pathao Courier", "RedX", "Paperfly",
    "Steadfast", "Karatoa", "QuickParcel BD", "SonarCourier", "DhakaExpress"
]
TELCO_BRANDS = [
    "Grameenphone", "GP", "Robi", "Banglalink", "Teletalk", "PadmaTel", "BongoNet"
]
ECOM_BRANDS = [
    "Daraz", "Shwapno", "Aarong", "Chaldal", "Pickaboo", "Rokomari",
    "MeghnaMart", "ShebaHaat", "BongoBazar"
]
EMPLOYER_BRANDS = [
    "Unilever Bangladesh", "Bdjobs", "GrameenTech", "Square Group", "Walton",
    "ApexDigital BD", "EarningHub BD", "DeshWorks"
]
CRYPTO_BRANDS = [
    "Binance", "CoinBangla", "BDTradeX", "AlphaCrypto", "DhakaSignals"
]
PERSON_NAMES = [
    "Rahim", "Karim", "Sadia", "Nusrat", "Tanvir", "Tariq", "Farhan", "Mitu", "Hasan", "Nabila"
]
PHISH_DOMAINS = [
    "verify-bd[.]top", "claim-now[.]xyz", "secure-kyc[.]site", "pay-fee[.]vip",
    "track-parcel[.]top", "bonus-bd[.]info", "vip-earn[.]xyz", "confirm-addr[.]top"
]
LEGIT_DOMAINS = [
    "daraz[.]com[.]bd/track", "payment[.]breb[.]org[.]bd", "mygp[.]grameenphone[.]com",
    "bkash[.]com/help", "sundarbancourier[.]com[.]bd", "bracbank[.]com/portal"
]


def _rand_amt(rng: random.Random) -> str:
    return rng.choice(["45", "50", "100", "250", "300", "500", "750", "1,000", "1,250", "1,500", "2,000", "2,500", "3,500", "5,000", "10,000", "50,000"])


def _rand_phone(rng: random.Random) -> str:
    prefix = rng.choice(["017", "018", "019", "016", "013", "015"])
    return f"{prefix}XXXXXX{rng.randint(10, 99)}"


def _rand_trx(rng: random.Random) -> str:
    chars = "ABCDEFGHJKLMNPQRSTUVWXYZ23456789"
    return "".join(rng.choice(chars) for _ in range(8))


def _rand_otp(rng: random.Random) -> str:
    return str(rng.randint(100000, 999999))


def _rand_parcel(rng: random.Random) -> str:
    return f"{rng.choice(['SP', 'SC', 'DZ', 'PF', 'CN', 'RX'])}{rng.randint(1000, 99999)}"


def _build_slots(rng: random.Random) -> dict:
    return {
        "mfs": rng.choice(MFS_BRANDS),
        "bank": rng.choice(BANK_BRANDS),
        "courier": rng.choice(COURIER_BRANDS),
        "telco": rng.choice(TELCO_BRANDS),
        "ecom": rng.choice(ECOM_BRANDS),
        "emp": rng.choice(EMPLOYER_BRANDS),
        "crypto": rng.choice(CRYPTO_BRANDS),
        "name": rng.choice(PERSON_NAMES),
        "amt": _rand_amt(rng),
        "bal": str(rng.randint(500, 25000)),
        "phone": _rand_phone(rng),
        "trx": _rand_trx(rng),
        "otp": _rand_otp(rng),
        "parcel": _rand_parcel(rng),
        "purl": f"hxxps://{rng.choice(PHISH_DOMAINS)}/{rng.choice(['verify', 'claim', 'pay', 'update'])}",
        "lurl": f"https://{rng.choice(LEGIT_DOMAINS)}",
    }


def _gen_single_templates(s: dict) -> list:
    """Return list of (family_id, language, label, techniques, action, is_hard_negative, text)."""
    rows = []

    # ==================== CLASS 0: LEGITIMATE (HARD LEGIT + PERSONAL) ====================
    # 1. Legit OTP with do-not-share warning (4 languages)
    rows.append(("c0_otp_warn", "english", 0, "NONE", 0, True,
                 f"{s['mfs']} OTP is {s['otp']}. Do not share this code with anyone, including {s['mfs']} staff. Valid 3 min."))
    rows.append(("c0_otp_warn", "bangla", 0, "NONE", 0, True,
                 f"আপনার {s['mfs']} ওটিপি {s['otp']}। এই কোড কারো সাথে শেয়ার করবেন না, এমনকি {s['mfs']} কর্মীকেও না।"))
    rows.append(("c0_otp_warn", "banglish", 0, "NONE", 0, True,
                 f"{s['bank']} OTP: {s['otp']}. Eta kauke bolben na. Bank er lokeo PIN ba OTP chay na. Meyad 3 minute."))
    rows.append(("c0_otp_warn", "mixed", 0, "NONE", 0, True,
                 f"{s['mfs']} login code {s['otp']} for Tk {s['amt']} payment. Kauke share korben na."))

    # 2. Legit Cash-out / Receive / Debit confirmation
    rows.append(("c0_tx_confirm", "english", 0, "NONE", 0, True,
                 f"{s['mfs']}: You have received Tk {s['amt']} from {s['phone']}. Fee Tk 0.00. Balance Tk {s['bal']}. TrxID {s['trx']}."))
    rows.append(("c0_tx_confirm", "bangla", 0, "NONE", 0, True,
                 f"আপনার {s['amt']} টাকা ক্যাশ আউট সফল হয়েছে। বর্তমান ব্যালেন্স {s['bal']} টাকা। TrxID {s['trx']}।"))
    rows.append(("c0_tx_confirm", "banglish", 0, "NONE", 0, True,
                 f"Dhonyobad! Apnar Tk {s['amt']} recharge successful. Balance Tk {s['bal']}. TrxID {s['trx']}."))
    rows.append(("c0_tx_confirm", "mixed", 0, "NONE", 0, True,
                 f"{s['bank']}: BDT {s['amt']} debited from A/C XXXX{s['otp'][:4]} at {s['ecom']}. Avl bal BDT {s['bal']}."))

    # 3. Legit Delivery / Parcel updates (no fee demanded)
    rows.append(("c0_parcel_legit", "english", 0, "NONE", 0, True,
                 f"Your {s['ecom']} order #{s['parcel']} is out for delivery today. Track: {s['lurl']}"))
    rows.append(("c0_parcel_legit", "bangla", 0, "NONE", 0, True,
                 f"{s['courier']}: আপনার পার্সেল {s['parcel']} আজ ডেলিভারি হবে। শাখা থেকে সংগ্রহ করতে এই এসএমএস ও এনআইডি আনুন।"))
    rows.append(("c0_parcel_legit", "banglish", 0, "NONE", 0, True,
                 f"{s['courier']}: Parcel {s['parcel']} Chattogram hub e pouchheche. Branch theke collect korun."))
    rows.append(("c0_parcel_legit", "mixed", 0, "NONE", 0, True,
                 f"{s['ecom']} order #{s['parcel']} delivered. Cash on delivery Tk {s['amt']} paid. Thank you."))

    # 4. Legit Security Alert / Scam Warning / Bill Reminder / Real Cashback Confirmation
    rows.append(("c0_sec_alert", "english", 0, "NONE", 0, True,
                 f"Security alert: New device login on {s['mfs']}. If this was you, ignore. We never ask for your PIN or OTP."))
    rows.append(("c0_sec_alert", "banglish", 0, "NONE", 0, True,
                 f"Sabdhan! Keu jodi {s['mfs']} agent bole PIN ba OTP chay, kokhono diben na. Eta scam."))
    rows.append(("c0_bill_remind", "mixed", 0, "NONE", 0, True,
                 f"Reminder: Your {s['telco']} bill of Tk {s['amt']} is due on 3 Dec. Pay via official app or {s['lurl']}"))
    rows.append(("c0_cashback_real", "bangla", 0, "NONE", 0, True,
                 f"আপনার {s['ecom']} পেমেন্টের জন্য Tk {s['amt']} ক্যাশব্যাক আপনার {s['mfs']} অ্যাকাউন্টে যোগ হয়েছে।"))

    # 5. Ordinary Personal Chat / Appointment
    rows.append(("c0_personal", "english", 0, "NONE", 0, False,
                 f"Hi {s['name']}, meeting moved to 3 PM. Please join the calendar invite link."))
    rows.append(("c0_personal", "bangla", 0, "NONE", 0, False,
                 f"{s['name']}, কাল সকাল ১০টায় ক্লাস হবে। স্যার ইমেইল করেছেন।"))
    rows.append(("c0_personal", "banglish", 0, "NONE", 0, False,
                 f"Mama, ajke raate bari asbo. Bazar theke kichu lagbe? {s['name']} ke bolo."))
    rows.append(("c0_personal", "mixed", 0, "NONE", 0, False,
                 f"Ami ekhon {s['mfs']} e {s['amt']} taka pathaisi, dekh to ashche kina."))

    # ==================== CLASS 1: MFS / OTP SCAM ====================
    # 1. Account lock / KYC threat asking for PIN/OTP/NID
    rows.append(("c1_kyc_lock", "english", 1, "T1|T2|T5", 1, True,
                 f"{s['mfs']} Helpline: Unusual login detected. Reply with your 6-digit OTP and PIN within 1 hour or account will be blocked."))
    rows.append(("c1_kyc_lock", "bangla", 1, "T1|T2|T5", 1, True,
                 f"প্রিয় গ্রাহক, আপনার {s['mfs']} অ্যাকাউন্ট ২৪ ঘণ্টার মধ্যে বন্ধ হয়ে যাবে। চালু রাখতে পিন ও ওটিপি নম্বর পাঠান।"))
    rows.append(("c1_kyc_lock", "banglish", 1, "T1|T2|T5", 1, True,
                 f"Dear {s['mfs']} user, KYC update na korle account band hobe. Update korte NID number o PIN pathan."))
    rows.append(("c1_kyc_lock", "mixed", 1, "T1|T2|T5", 1, True,
                 f"{s['bank']} Security: Card ending {s['otp'][:4]} flagged. Share the OTP you just received. Last warning."))

    # 2. Mistaken transfer claim asking for money back / PIN
    rows.append(("c1_mistake_tx", "banglish", 1, "T1|T2|T5", 1, True,
                 f"Ami {s['mfs']} agent, bhul number-e {s['amt']} taka pathaisi. Ferot din, PIN ta bolen."))
    rows.append(("c1_mistake_tx", "mixed", 1, "T3|T5", 2, True,
                 f"Bhai ekta bhul hoye geche. Amar {s['amt']} taka vul kore apnar {s['mfs']} e chole gese. Please ferot den {s['phone']} e. Ami bipode."))
    rows.append(("c1_mistake_tx", "bangla", 1, "T1|T2|T5", 1, True,
                 f"আমি {s['mfs']} এজেন্ট, ভুল করে আপনার নম্বরে {s['amt']} টাকা গেছে। টাকা ফেরত দিতে ওটিপি কোডটি বলুন।"))
    rows.append(("c1_mistake_tx", "english", 1, "T1|T3|T5", 2, True,
                 f"Mistaken transfer of Tk {s['amt']} sent to your {s['mfs']}. Return immediately to {s['phone']} or police action will be taken."))

    # 3. Card / Refund team phishing
    rows.append(("c1_card_refund", "english", 1, "T1|T2", 1, False,
                 f"{s['ecom']} refund team: send card number, expiry and CVV to receive your Tk {s['amt']} refund."))
    rows.append(("c1_card_refund", "banglish", 1, "T1|T2", 1, False,
                 f"{s['mfs']} bonus: apni Tk {s['amt']} peyechen. Wallet e add korte PIN o card CVV din."))

    # ==================== CLASS 2: PRIZE / LOTTERY SCAM ====================
    rows.append(("c2_lottery_link", "english", 2, "T4|T5", 3, False,
                 f"Congratulations! You won Tk {s['amt']} cashback. Click {s['purl']} to claim within 1 hour."))
    rows.append(("c2_lottery_link", "bangla", 2, "T4|T5", 3, False,
                 f"অভিনন্দন! আপনি {s['amt']} টাকা লটারি জিতেছেন। পুরস্কার পেতে এখনই লিংকে ক্লিক করুন: {s['purl']}"))
    rows.append(("c2_lottery_link", "banglish", 2, "T4|T5", 3, False,
                 f"Apni {s['amt']} taka jitechen! Claim korte link e click korun: {s['purl']}"))
    rows.append(("c2_lottery_link", "mixed", 2, "T1|T4|T5", 3, True,
                 f"Free 10GB data gift from {s['telco']} for you! Tap {s['purl']} now, offer expires in 1 hour."))

    rows.append(("c2_prize_fee", "english", 2, "T1|T3|T4", 2, False,
                 f"You won Tk {s['amt']} from {s['telco']} lucky draw! Pay Tk 300 processing fee via {s['mfs']} to claim."))
    rows.append(("c2_prize_fee", "banglish", 2, "T1|T4", 5, False,
                 f"Congratulations! Apnar number ti {s['telco']} Eid lucky draw te select hoyeche. iPhone jitar jonno {s['phone']} e WhatsApp korun."))
    rows.append(("c2_prize_app", "mixed", 2, "T1|T4", 4, False,
                 f"{s['mfs']} anniversary: spin and win Tk {s['amt']}! Download the spin app: {s['purl']}/app.apk"))
    rows.append(("c2_prize_details", "english", 2, "T4", 1, False,
                 f"Lucky draw winner! Reply with your full name, NID and address to receive a brand new car."))

    # ==================== CLASS 3: FAKE JOB / INVESTMENT ====================
    rows.append(("c3_task_earn", "english", 3, "T6", 4, False,
                 f"Part-time job: like YouTube videos, earn Tk {s['amt']} per task. Join Telegram: {s['purl']}"))
    rows.append(("c3_task_earn", "bangla", 3, "T6", 4, False,
                 f"ঘরে বসে প্রতিদিন {s['amt']} টাকা আয় করুন। সহজ লাইক ও কমেন্টের কাজ। আজই হোয়াটসঅ্যাপে যোগ দিন।"))
    rows.append(("c3_task_earn", "banglish", 3, "T6", 4, False,
                 f"Ghor bose din e {s['amt']} taka income. Facebook page like korle taka. WhatsApp e join korun."))
    rows.append(("c3_task_earn", "mixed", 3, "T6", 4, False,
                 f"Task: buy products on {s['ecom']} via our link, we refund + 20% commission. Complete 5 tasks, get Tk {s['amt']}."))

    rows.append(("c3_invest_fee", "english", 3, "T3|T6", 2, False,
                 f"Double your money in 7 days! Invest Tk {s['amt']}, get double guaranteed. Send via {s['mfs']} {s['phone']}."))
    rows.append(("c3_invest_fee", "mixed", 3, "T1|T3|T6", 2, False,
                 f"Congratulations, selected for Data Entry Officer at {s['emp']}. Pay Tk {s['amt']} registration fee to confirm."))
    rows.append(("c3_crypto", "english", 3, "T5|T6", 4, False,
                 f"{s['crypto']} VIP signals group - 300% weekly profit guaranteed! Join now: {s['purl']}"))
    rows.append(("c3_abroad_job", "banglish", 3, "T3|T6", 5, False,
                 f"Abroad job in Malaysia, visa fee Tk {s['amt']}. 100% guaranteed. Call {s['phone']} ekhoni."))

    # ==================== CLASS 4: DELIVERY / PARCEL PHISHING ====================
    rows.append(("c4_parcel_fee", "english", 4, "T1|T3", 2, True,
                 f"{s['courier']}: Parcel {s['parcel']} held at hub. Pay Tk {s['amt']} storage fee: {s['purl']}"))
    rows.append(("c4_parcel_fee", "bangla", 4, "T3", 2, True,
                 f"আপনার পার্সেল {s['parcel']} আটকে আছে। ডেলিভারি চার্জ {s['amt']} টাকা পরিশোধ করতে লিংকে ক্লিক করুন: {s['purl']}"))
    rows.append(("c4_parcel_fee", "banglish", 4, "T3", 2, True,
                 f"Parcel {s['parcel']} atke ase. Delivery fee Tk {s['amt']} dite link e jan: {s['purl']}"))
    rows.append(("c4_customs", "mixed", 4, "T1|T3", 2, True,
                 f"Customs hold: your parcel from China needs Tk {s['amt']} duty. Pay via {s['mfs']} {s['phone']} to release."))

    rows.append(("c4_addr_link", "english", 4, "T1|T5", 3, True,
                 f"Courier notice: package #{s['parcel']} requires address confirmation. Click {s['purl']} within 24h."))
    rows.append(("c4_addr_link", "bangla", 4, "T1|T5", 3, True,
                 f"{s['courier']}: ডেলিভারি ব্যর্থ হয়েছে। ঠিকানা আপডেট করতে ১২ ঘণ্টার মধ্যে {s['purl']} এ যান।"))
    rows.append(("c4_addr_call", "banglish", 4, "T1|T5", 5, True,
                 f"{s['courier']}: Apnar parcel er address vul. Call korun {s['phone']} ekhoni."))

    # ==================== CLASS 5: PROMOTIONAL SPAM ====================
    rows.append(("c5_promo_offer", "english", 5, "NONE", 0, True,
                 f"Get 10% cashback on your next top-up with code EID10. Valid today only!"))
    rows.append(("c5_promo_offer", "bangla", 5, "NONE", 0, True,
                 f"ঈদ উপলক্ষে {s['ecom']}-এ সব পণ্যে ৩০% ছাড়! আজই কিনুন, শর্ত প্রযোজ্য।"))
    rows.append(("c5_promo_offer", "banglish", 5, "NONE", 0, True,
                 f"Eid offer! {s['ecom']} e sob item e 30% discount. Shop now: {s['lurl']}"))
    rows.append(("c5_promo_offer", "mixed", 5, "NONE", 0, True,
                 f"{s['telco']} Mega Offer! Buy 1GB get 1GB free for Tk {s['amt']}. Dial *121*3#"))
    rows.append(("c5_promo_app", "english", 5, "NONE", 0, False,
                 f"Install {s['ecom']} app and get Tk 50 off your first order. Use code WELCOME50."))
    rows.append(("c5_promo_biz", "mixed", 5, "NONE", 0, False,
                 f"Book your dream apartment in Bashundhara, easy installment. Call {s['phone']}"))
    rows.append(("c5_promo_course", "english", 5, "NONE", 0, False,
                 f"Learn English in 30 days! Admission going on at {s['emp']}. Call now."))

    return rows


def _gen_multiturn_templates(rng: random.Random, s: dict) -> list:
    """Generate multi-turn conversations (2-12 turns) with delayed intent, small talk, or strategy shifts."""
    rows = []
    spk_s, spk_r = rng.choice([("S", "R"), ("Caller", "You"), ("Agent", "Customer"), ("A", "B")])

    # Class 0 multi-turn: personal chat or legit support
    t_c0_1 = [
        f"{spk_s}: Bhai, kemon asos?",
        f"{spk_r}: Alhamdulillah bhalo. Tui?",
        f"{spk_s}: Bhalo. {s['amt']} taka lagbe, Sunday ferot dibo?",
        f"{spk_r}: Ha thik ase, {s['mfs']} e pathacchi.",
        f"{spk_s}: Ok thanks!"
    ]
    rows.append(("c0_mt_chat", "banglish", 0, "NONE", 0, True, len(t_c0_1), " ||| ".join(t_c0_1)))

    t_c0_2 = [
        f"{spk_r}: I forgot my {s['mfs']} PIN. How can I reset it?",
        f"{spk_s}: Dial *247# and choose Reset PIN.",
        f"{spk_s}: Never share your PIN or OTP with anyone, including our support staff.",
        f"{spk_r}: Thank you!"
    ]
    rows.append(("c0_mt_support", "english", 0, "NONE", 0, True, len(t_c0_2), " ||| ".join(t_c0_2)))

    t_c0_3 = [
        f"{spk_r}: আমার পার্সেল {s['parcel']} কি আজ আসবে?",
        f"{spk_s}: হ্যাঁ, রাইডার বিকেল ৫টার দিকে ডেলিভারি করবে।",
        f"{spk_r}: ঠিক আছে, আমি ক্যাশ অন ডেলিভারি দেবো।"
    ]
    rows.append(("c0_mt_delivery", "bangla", 0, "NONE", 0, True, len(t_c0_3), " ||| ".join(t_c0_3)))

    # Class 1 multi-turn: trust building -> mistaken transfer -> PIN/OTP ask (up to 7 turns)
    t_c1_1 = [
        f"{spk_s}: Assalamu alaikum, ami {s['mfs']} office theke bolchi.",
        f"{spk_r}: Wa alaikumussalam, bolun.",
        f"{spk_s}: Apnar account e technical check cholche. Bhul kore {s['amt']} taka chole gese.",
        f"{spk_r}: Ami to kono SMS pai nai.",
        f"{spk_s}: Wait korun, ekta 6 digit code asbe.",
        f"{spk_s}: Account verify korte oi OTP ar apnar PIN ta bolen, na hole account block hobe."
    ]
    rows.append(("c1_mt_agent", "banglish", 1, "T1|T2|T5", 1, True, len(t_c1_1), " ||| ".join(t_c1_1)))

    t_c1_2 = [
        f"{spk_s}: Hello, I am calling from {s['bank']} security.",
        f"{spk_r}: Yes, is there an issue?",
        f"{spk_s}: Someone tried to withdraw Tk {s['amt']} from your card.",
        f"{spk_r}: Oh no! What should I do?",
        f"{spk_s}: To cancel it, quickly tell me the OTP you just received and your card CVV."
    ]
    rows.append(("c1_mt_bank", "english", 1, "T1|T2|T5", 1, True, len(t_c1_2), " ||| ".join(t_c1_2)))

    # Class 2 multi-turn: prize bait -> fee or link
    t_c2_1 = [
        f"{spk_s}: Hello {s['name']}, you have been selected for our {s['ecom']} loyalty reward!",
        f"{spk_r}: Really? What reward?",
        f"{spk_s}: A free smartphone worth 45000 BDT.",
        f"{spk_r}: How do I get it?",
        f"{spk_s}: Just pay Tk {s['amt']} delivery charge via {s['mfs']} to {s['phone']}."
    ]
    rows.append(("c2_mt_prize", "english", 2, "T1|T3|T4", 2, False, len(t_c2_1), " ||| ".join(t_c2_1)))

    t_c2_2 = [
        f"{spk_s}: অভিনন্দন! আপনি {s['telco']} লটারিতে {s['amt']} টাকা জিতেছেন।",
        f"{spk_r}: সত্যি নাকি? কীভাবে পাবো?",
        f"{spk_s}: এই লিংকে গিয়ে ফর্ম পূরণ করুন: {s['purl']}। ১ ঘণ্টার মধ্যে ক্লেইম করুন।"
    ]
    rows.append(("c2_mt_lottery_bn", "bangla", 2, "T1|T4|T5", 3, False, len(t_c2_2), " ||| ".join(t_c2_2)))

    t_c4_2 = [
        f"{spk_s}: আপনার {s['courier']} পার্সেল #{s['parcel']} কাস্টমসে আটকে আছে।",
        f"{spk_r}: আমি তো কিছু অর্ডার করিনি।",
        f"{spk_s}: আপনার নামে এসেছে। ছাড়াতে {s['amt']} টাকা ফি দিন: {s['purl']}"
    ]
    rows.append(("c4_mt_customs_bn", "bangla", 4, "T1|T3|T5", 2, True, len(t_c4_2), " ||| ".join(t_c4_2)))

    t_c5_2 = [
        f"{spk_s}: {s['ecom']} মেগা সেলে আজ সব পণ্যে ৪০% ছাড়!",
        f"{spk_s}: অফার নিতে ভিজিট করুন {s['lurl']}"
    ]
    rows.append(("c5_mt_promo_bn", "bangla", 5, "NONE", 0, True, len(t_c5_2), " ||| ".join(t_c5_2)))

    t_c1_3 = [
        f"{spk_s}: হ্যালো, আমি {s['mfs']} কাস্টমার কেয়ার থেকে বলছি।",
        f"{spk_r}: জ্বী বলুন।",
        f"{spk_s}: আপনার একাউন্টে সমস্যা হয়েছে, দ্রুত আপনার পিন ও ওটিপি দিন।"
    ]
    rows.append(("c1_mt_agent_bn", "bangla", 1, "T1|T2|T5", 1, True, len(t_c1_3), " ||| ".join(t_c1_3)))

    t_c0_4 = [
        f"{spk_r}: Bhai, {s['ecom']} order #{s['parcel']} ki ashche?",
        f"{spk_s}: Ha, rider call korse, 5 tay delivery hobe.",
        f"{spk_r}: Ok, cash on delivery dibo."
    ]
    rows.append(("c0_mt_delivery_mix", "mixed", 0, "NONE", 0, True, len(t_c0_4), " ||| ".join(t_c0_4)))

    t_c4_3 = [
        f"{spk_s}: {s['courier']}: Parcel #{s['parcel']} address incomplete.",
        f"{spk_r}: Ki korte hobe?",
        f"{spk_s}: Confirm address within 12h at {s['purl']} or parcel return hobe."
    ]
    rows.append(("c4_mt_addr_mix", "mixed", 4, "T1|T5", 3, True, len(t_c4_3), " ||| ".join(t_c4_3)))

    # Class 3 multi-turn: job interview -> strategy shift to registration fee / Telegram task
    t_c3_1 = [
        f"{spk_s}: Hi, I saw your CV on {s['emp']}. We have a work-from-home position.",
        f"{spk_r}: What is the job and salary?",
        f"{spk_s}: Data entry, 35,000 BDT per month.",
        f"{spk_r}: Sounds good, how do I start?",
        f"{spk_s}: You need to deposit Tk {s['amt']} as security/registration fee first."
    ]
    rows.append(("c3_mt_jobfee", "english", 3, "T1|T3|T6", 2, False, len(t_c3_1), " ||| ".join(t_c3_1)))

    t_c3_2 = [
        f"{spk_s}: Hello, part-time online kaj korte chan?",
        f"{spk_r}: Ki kaj?",
        f"{spk_s}: YouTube video like korle daily {s['amt']} taka income.",
        f"{spk_s}: First e Telegram group e join korun: {s['purl']}"
    ]
    rows.append(("c3_mt_task", "mixed", 3, "T6", 4, False, len(t_c3_2), " ||| ".join(t_c3_2)))

    # Class 4 multi-turn: parcel notice -> customs fee or address link
    t_c4_1 = [
        f"{spk_s}: Hello, this is {s['courier']}. You have a package #{s['parcel']} from abroad.",
        f"{spk_r}: I didn't order anything.",
        f"{spk_s}: It is registered under your name at customs.",
        f"{spk_r}: What do I need to do?",
        f"{spk_s}: Pay Tk {s['amt']} customs duty today or it will be returned."
    ]
    rows.append(("c4_mt_customs", "english", 4, "T1|T3|T5", 2, True, len(t_c4_1), " ||| ".join(t_c4_1)))

    # Class 5 multi-turn: promo dialogue
    t_c5_1 = [
        f"{spk_s}: Sir, hair fall problem? Our herbal oil is very popular.",
        f"{spk_s}: 40% discount today only, order now at {s['lurl']}."
    ]
    rows.append(("c5_mt_promo", "english", 5, "NONE", 0, True, len(t_c5_1), " ||| ".join(t_c5_1)))

    # Longer 8-10 turn conversation (Class 1 or Class 3) to cover up to 12 turns
    t_long = [
        f"{spk_s}: Assalamu alaikum, apnar sathe kotha bola jabe?",
        f"{spk_r}: Ji bolun, ke bolchen?",
        f"{spk_s}: Ami {s['emp']} HR theke {s['name']}.",
        f"{spk_r}: Oh accha, ki bishoye?",
        f"{spk_s}: Apni amader online task program e select hoyechen.",
        f"{spk_r}: Eta ki rokom kaj?",
        f"{spk_s}: Ghor bose daily {s['amt']} taka income kora jabe.",
        f"{spk_r}: Kono fee lagbe?",
        f"{spk_s}: Ji, shudhu {s['amt']} taka registration fee diye WhatsApp group e join korte hobe."
    ]
    rows.append(("c3_mt_long", "banglish", 3, "T1|T3|T6", 4, False, len(t_long), " ||| ".join(t_long)))

    return rows


def generate_synthetic_dataset(target_rows: int = 6000, seed: int = 42) -> pd.DataFrame:
    rng = random.Random(seed)
    # Target counts per class (extra weight for 0 and 5 as requested)
    target_per_class = {
        0: int(target_rows * 0.20),  # 1200
        1: int(target_rows * 0.155), # 930
        2: int(target_rows * 0.155), # 930
        3: int(target_rows * 0.155), # 930
        4: int(target_rows * 0.155), # 930
        5: int(target_rows * 0.18),  # 1080
    }

    buckets = {k: [] for k in range(6)}

    # Keep generating batches until all class buckets are full
    while any(len(buckets[k]) < target_per_class[k] for k in range(6)):
        slots = _build_slots(rng)
        single_rows = _gen_single_templates(slots)
        mt_rows = _gen_multiturn_templates(rng, slots)

        # ~65% single, ~35% multi-turn weighting, boosting non-English slightly for ~25% language balance
        for fam, lang, lbl, tech, act, hard, txt in single_rows:
            lang_p = 0.40 if lang == "english" else 0.72
            if len(buckets[lbl]) < target_per_class[lbl] and rng.random() < lang_p:
                buckets[lbl].append({
                    "text": txt,
                    "label": lbl,
                    "techniques": "|".join(sorted(tech.split("|"))) if tech != "NONE" else "NONE",
                    "action": act,
                    "family_id": fam,
                    "language": lang,
                    "n_turns": 1,
                    "is_hard_negative": hard,
                })

        for fam, lang, lbl, tech, act, hard, n_t, txt in mt_rows:
            if len(buckets[lbl]) < target_per_class[lbl] and rng.random() < 0.85:
                buckets[lbl].append({
                    "text": txt,
                    "label": lbl,
                    "techniques": "|".join(sorted(tech.split("|"))) if tech != "NONE" else "NONE",
                    "action": act,
                    "family_id": fam,
                    "language": lang,
                    "n_turns": n_t,
                    "is_hard_negative": hard,
                })

    all_rows = []
    for k in range(6):
        all_rows.extend(buckets[k][:target_per_class[k]])

    rng.shuffle(all_rows)
    df = pd.DataFrame(all_rows)
    df.insert(0, "id", range(100001, 100001 + len(df)))
    return df

import os
import random
import re
import pandas as pd


# Map v1 family_ids to scenario_id
V1_FAMILY_TO_SCENARIO = {
    "c0_otp_warn": "0_otp_genuine_notice",
    "c0_tx_confirm": "0_tx_confirmation",
    "c0_parcel_legit": "0_delivery_genuine",
    "c0_mt_delivery": "0_delivery_genuine",
    "c0_mt_delivery_mix": "0_delivery_genuine",
    "c0_sec_alert": "0_security_warning",
    "c0_mt_support": "0_security_warning",
    "c0_bill_remind": "0_bill_reminder",
    "c0_cashback_real": "0_cashback_genuine",
    "c0_personal": "0_personal_chat",
    "c0_mt_chat": "0_personal_chat",
    "c1_kyc_lock": "1_account_lock_kyc_pin",
    "c1_mt_bank": "1_account_lock_kyc_pin",
    "c1_mt_agent_bn": "1_account_lock_kyc_pin",
    "c1_mistake_tx": "1_mistaken_transfer",
    "c1_mt_agent": "1_mistaken_transfer",
    "c1_card_refund": "1_card_refund_cvv",
    "c2_lottery_link": "2_lottery_cashback_link",
    "c2_mt_lottery_bn": "2_lottery_cashback_link",
    "c2_prize_fee": "2_prize_advance_fee",
    "c2_mt_prize": "2_prize_advance_fee",
    "c2_prize_app": "2_prize_spin_apk",
    "c2_prize_details": "2_prize_details_reply",
    "c3_task_earn": "3_task_earning",
    "c3_mt_task": "3_task_earning",
    "c3_invest_fee": "3_investment_job_fee",
    "c3_mt_jobfee": "3_investment_job_fee",
    "c3_mt_long": "3_investment_job_fee",
    "c3_crypto": "3_crypto_vip_signals",
    "c3_abroad_job": "3_abroad_visa_fee",
    "c4_parcel_fee": "4_parcel_storage_fee",
    "c4_customs": "4_parcel_customs_fee",
    "c4_mt_customs": "4_parcel_customs_fee",
    "c4_mt_customs_bn": "4_parcel_customs_fee",
    "c4_addr_link": "4_parcel_address_link",
    "c4_mt_addr_mix": "4_parcel_address_link",
    "c4_addr_call": "4_parcel_address_call",
    "c5_promo_offer": "5_promo_sales_offer",
    "c5_mt_promo": "5_promo_sales_offer",
    "c5_mt_promo_bn": "5_promo_sales_offer",
    "c5_promo_app": "5_promo_app_install",
    "c5_promo_biz": "5_promo_real_estate",
    "c5_promo_course": "5_promo_course_ad",
}


def _wrap_multiturn(rng: random.Random, s: dict, body_turns: list, lead_in_style: str = "neutral"):
    spk_s, spk_r = rng.choice([("S", "R"), ("Caller", "You"), ("Agent", "Customer"), ("A", "B")])
    if lead_in_style == "greet":
        intro = [
            f"{spk_s}: Assalamu alaikum {s['name']}, kemon achen?",
            f"{spk_r}: Wa alaikumussalam, bhalo. Ke bolchen?",
        ]
    elif lead_in_style == "casual":
        intro = [
            f"{spk_s}: Hello, are you free for a moment?",
            f"{spk_r}: Yes, who is this?",
        ]
    elif lead_in_style == "bangla":
        intro = [
            f"{spk_s}: হ্যালো, একটু কথা বলা যাবে?",
            f"{spk_r}: জ্বী বলুন, কে বলছেন?",
        ]
    else:
        intro = []
    formatted = intro + [f"{spk_s if i % 2 == 0 else spk_r}: {t}" for i, t in enumerate(body_turns)]
    return len(formatted), " ||| ".join(formatted)


def _gen_new_scenarios(rng: random.Random, s: dict) -> list:
    """Return list of (scenario_id, family_id, language, label, techniques, action, n_turns, is_hard_negative, text)."""
    rows = []

    def add(scen, lang, lbl, tech, act, hard, txt, n_t=1):
        rows.append((scen, scen, lang, lbl, tech, act, n_t, hard, txt))

    # =========================================================================
    # CLASS 0: 14 NEW SCENARIOS (+ 7 v1 = 21 scenarios total)
    # =========================================================================
    # 0_work_chat
    add("0_work_chat", "english", 0, "NONE", 0, False,
        f"Team update: Sprint review moved to 4 PM by {s['name']}. Please push your branch before then.")
    add("0_work_chat", "banglish", 0, "NONE", 0, False,
        f"{s['name']}, client er presentation deck ta ajke dupurer moddhe mail kore dio.")
    add("0_work_chat", "bangla", 0, "NONE", 0, False,
        f"{s['name']}, অফিসের রিপোর্টটা রেডি হলে গুগল ড্রাইভে আপলোড করে দিও।")
    n_t, mt = _wrap_multiturn(rng, s, [
        "Did you check the server logs from last night?",
        "Yes, latency spiked around 2 AM, already patched.",
        "Great, let's monitor for an hour."
    ], "casual")
    add("0_work_chat", "mixed", 0, "NONE", 0, False, mt, n_t)

    # 0_appointment_reminder
    add("0_appointment_reminder", "english", 0, "NONE", 0, False,
        f"Clinic Reminder: Your dental checkup with Dr. {s['name']} is tomorrow at 11:30 AM. Reply C to confirm.")
    add("0_appointment_reminder", "bangla", 0, "NONE", 0, False,
        f"হাসপাতাল নোটিশ: আগামীকাল সকাল ১০টায় আপনার অ্যাপয়েন্টমেন্ট নির্ধারিত আছে।")
    add("0_appointment_reminder", "banglish", 0, "NONE", 0, False,
        f"Diagnostic center: Apnar blood test report ready. Kal bikal 5 tay collect korte parben.")
    add("0_appointment_reminder", "mixed", 0, "NONE", 0, False,
        f"Passport office appointment confirmed for 12 Dec at 9 AM. Bring original NID.")

    # 0_bank_statement
    add("0_bank_statement", "english", 0, "NONE", 0, True,
        f"{s['bank']}: Monthly e-statement for A/C XX{s['otp'][:4]} is ready. Closing balance BDT {s['bal']}. View in official app.")
    add("0_bank_statement", "bangla", 0, "NONE", 0, True,
        f"{s['bank']}: আপনার মাসিক ব্যাংক স্টেটমেন্ট তৈরি হয়েছে। ব্যালেন্স {s['bal']} টাকা।")
    add("0_bank_statement", "banglish", 0, "NONE", 0, True,
        f"{s['bank']} card statement generated. Total due Tk {s['amt']}, min due Tk 500. Pay via app.")
    add("0_bank_statement", "mixed", 0, "NONE", 0, True,
        f"{s['bank']}: Your FDR interest of Tk {s['amt']} has been credited. Avl bal Tk {s['bal']}.")

    # 0_service_notice_link (hard legit with link)
    add("0_service_notice_link", "english", 0, "NONE", 0, True,
        f"{s['telco']} Notice: Scheduled maintenance tonight 2 AM - 4 AM. Status updates: {s['lurl']}")
    add("0_service_notice_link", "bangla", 0, "NONE", 0, True,
        f"{s['mfs']} সেবা বিজ্ঞপ্তি: আজ রাত ২টা থেকে ৪টা পর্যন্ত অ্যাপ রক্ষণাবেক্ষণ চলবে। বিস্তারিত: {s['lurl']}")
    add("0_service_notice_link", "banglish", 0, "NONE", 0, True,
        f"DESCO bill copy download korte parben ekhan theke: {s['lurl']}. Kono fee lagbe na.")
    add("0_service_notice_link", "mixed", 0, "NONE", 0, True,
        f"{s['ecom']} privacy policy updated. Read details at {s['lurl']}.")

    # 0_family_transfer_chat
    n_t, mt = _wrap_multiturn(rng, s, [
        f"Ammu, bazar er jonno Tk {s['amt']} pathalam {s['mfs']} e.",
        "Accha baba, check korchi. Balance asche.",
        "Thik ase, lagle janio."
    ], "greet")
    add("0_family_transfer_chat", "banglish", 0, "NONE", 0, True, mt, n_t)
    add("0_family_transfer_chat", "bangla", 0, "NONE", 0, True,
        f"আব্বু, তোমার {s['mfs']} নম্বরে {s['amt']} টাকা পাঠিয়েছি, পেয়েছো কিনা জানিও।")
    add("0_family_transfer_chat", "english", 0, "NONE", 0, True,
        f"Hey sis, I sent Tk {s['amt']} to your {s['mfs']} for mom's medicine. Let me know when you get it.")
    add("0_family_transfer_chat", "mixed", 0, "NONE", 0, True,
        f"Bhai, dinner bill er {s['amt']} taka tor {s['mfs']} e diye disi.")

    # 0_kyc_branch_notice (hard negative twin of 1_kyc_lock)
    add("0_kyc_branch_notice", "english", 0, "NONE", 0, True,
        f"{s['bank']}: Please update your KYC by visiting your nearest branch with original NID. Never share PIN or OTP via phone.")
    add("0_kyc_branch_notice", "bangla", 0, "NONE", 0, True,
        f"{s['bank']}: আপনার কেওয়াইসি হালনাগাদ করতে নিকটস্থ শাখায় এনআইডি নিয়ে আসুন। ফোনে পিন বা ওটিপি দেবেন না।")
    add("0_kyc_branch_notice", "banglish", 0, "NONE", 0, True,
        f"{s['mfs']} notice: KYC verify korte shudhu official app use korun. Agent ke PIN diben na.")
    add("0_kyc_branch_notice", "mixed", 0, "NONE", 0, True,
        f"SIM biometric re-verification: visit {s['telco']} customer care center with NID. No link or fee required.")

    # 0_self_pwd_reset (hard negative twin of 1_pwd_reset_link)
    add("0_self_pwd_reset", "english", 0, "NONE", 0, True,
        f"You requested a password reset on {s['ecom']}. If you did not request this, ignore this message. Do not share your code.")
    add("0_self_pwd_reset", "banglish", 0, "NONE", 0, True,
        f"{s['mfs']} PIN reset request peyeche. Apni na korle 16247 e call korun. Code kauke bolben na.")
    add("0_self_pwd_reset", "bangla", 0, "NONE", 0, True,
        f"আপনি পাসওয়ার্ড পরিবর্তনের অনুরোধ করেছেন। না করে থাকলে এটি উপেক্ষা করুন, কোড শেয়ার করবেন না।")
    add("0_self_pwd_reset", "mixed", 0, "NONE", 0, True,
        f"Login alert: {s['bank']} app password changed successfully. Contact branch if unauthorized.")

    # 0_cod_delivery_legit (hard negative twin of 4_cod_confirm)
    add("0_cod_delivery_legit", "english", 0, "NONE", 0, True,
        f"{s['ecom']} Rider {s['name']} is on the way with order #{s['parcel']}. Pay Tk {s['amt']} cash on delivery upon receiving.")
    add("0_cod_delivery_legit", "bangla", 0, "NONE", 0, True,
        f"আপনার অর্ডার #{s['parcel']} ডেলিভারির জন্য বের হয়েছে। পণ্য হাতে পেয়ে {s['amt']} টাকা ক্যাশ দেবেন।")
    add("0_cod_delivery_legit", "banglish", 0, "NONE", 0, True,
        f"{s['courier']}: Rider {s['name']} apnar parcel niye asche. Cash on delivery Tk {s['amt']}.")
    n_t, mt = _wrap_multiturn(rng, s, [
        f"Sir, ami {s['courier']} rider bolchi, apnar parcel niye gate e aschi.",
        "Accha ashen, cash on delivery koto?",
        f"Tk {s['amt']} sir, packet check kore taka diyen."
    ], "neutral")
    add("0_cod_delivery_legit", "mixed", 0, "NONE", 0, True, mt, n_t)

    # 0_emi_reminder
    add("0_emi_reminder", "english", 0, "NONE", 0, True,
        f"{s['bank']}: Reminder that your auto-debit EMI of Tk {s['amt']} is scheduled on 5th of this month.")
    add("0_emi_reminder", "bangla", 0, "NONE", 0, True,
        f"{s['bank']}: আপনার মাসিক কিস্তি {s['amt']} টাকা আগামী ৫ তারিখ স্বয়ংক্রিয়ভাবে কাটা হবে।")
    add("0_emi_reminder", "banglish", 0, "NONE", 0, True,
        f"{s['bank']} loan EMI Tk {s['amt']} due soon. Account e balance rakhun.")
    add("0_emi_reminder", "mixed", 0, "NONE", 0, True,
        f"Credit card bill payment of Tk {s['amt']} received by {s['bank']}. Thank you.")

    # 0_govt_portal_legit
    add("0_govt_portal_legit", "english", 0, "NONE", 0, False,
        "NID Service: Your smart NID card is ready for collection at your Upazila Election Office.")
    add("0_govt_portal_legit", "bangla", 0, "NONE", 0, False,
        "নির্বাচন কমিশন: আপনার স্মার্ট এনআইডি কার্ড উপজেলা অফিসে এসেছে। মূল স্লিপসহ সংগ্রহ করুন।")
    add("0_govt_portal_legit", "banglish", 0, "NONE", 0, False,
        "BRTA notice: Apnar driving license card print complete. Office theke collect korun.")
    add("0_govt_portal_legit", "mixed", 0, "NONE", 0, False,
        "NBR e-TIN confirmation: Your tax return acknowledgement slip is available in portal.")

    # 0_telco_pack_expiry
    add("0_telco_pack_expiry", "english", 0, "NONE", 0, True,
        f"{s['telco']}: Your 5GB internet pack expires tonight at 11:59 PM. Dial *121# or open app to renew.")
    add("0_telco_pack_expiry", "bangla", 0, "NONE", 0, True,
        f"{s['telco']}: আপনার ইন্টারনেট প্যাকের মেয়াদ আজ শেষ হবে। রিনিউ করতে অফিসিয়াল অ্যাপ দেখুন।")
    add("0_telco_pack_expiry", "banglish", 0, "NONE", 0, True,
        f"Dear customer, apnar {s['telco']} minute bundle sesh hoye jabe ajke. Dial *123*1#.")
    add("0_telco_pack_expiry", "mixed", 0, "NONE", 0, True,
        f"{s['telco']} balance low: remaining Tk 4.20. Recharge via {s['mfs']} or official app.")

    # 0_subscription_renewal
    add("0_subscription_renewal", "english", 0, "NONE", 0, False,
        f"Your cloud storage subscription (Tk {s['amt']}/month) has been renewed. Receipt sent to email.")
    add("0_subscription_renewal", "bangla", 0, "NONE", 0, False,
        f"আপনার মাসিক নিউজ অ্যাপ সাবস্ক্রিপশন সফলভাবে নবায়ন হয়েছে।")
    add("0_subscription_renewal", "banglish", 0, "NONE", 0, False,
        f"WiFi bill Tk {s['amt']} paid to {s['telco']}. Connection active for next 30 days.")
    add("0_subscription_renewal", "mixed", 0, "NONE", 0, False,
        f"Spotify / OTT monthly plan renewed for Tk {s['amt']}. Manage in account settings.")

    # 0_interview_legit (hard negative twin of 3_data_entry_job)
    add("0_interview_legit", "english", 0, "NONE", 0, True,
        f"HR, {s['emp']}: Your interview is scheduled Tuesday 10 AM at our Gulshan office. Bring CV. No fee required.")
    add("0_interview_legit", "bangla", 0, "NONE", 0, True,
        f"{s['emp']} এইচআর: আগামী মঙ্গলবার সকাল ১০টায় আপনার চাকরির সাক্ষাৎকার। সিভি সাথে আনবেন।")
    add("0_interview_legit", "banglish", 0, "NONE", 0, True,
        f"{s['emp']} office theke bolchi: apnar CV shortlist hoyeche, Thursday 11 AM e interview.")
    add("0_interview_legit", "mixed", 0, "NONE", 0, True,
        f"Walk-in interview at {s['emp']} Dhaka HQ on Sunday. Bring NID copy and certificates.")

    # 0_exam_university
    add("0_exam_university", "english", 0, "NONE", 0, False,
        "University Registrar: Final exam routine for Fall semester has been published on the student portal.")
    add("0_exam_university", "bangla", 0, "NONE", 0, False,
        "ক্লাস নোটিশ: আগামীকাল সকাল ৯টার ক্লাস ১০টায় অনুষ্ঠিত হবে।")
    add("0_exam_university", "banglish", 0, "NONE", 0, False,
        f"Bhai {s['name']}, kal er lab report ar assignment er deadline ki Friday?")
    add("0_exam_university", "mixed", 0, "NONE", 0, False,
        "Library notice: The book you reserved is ready for pickup within 2 days.")

    # =========================================================================
    # CLASS 1: 13 NEW SCENARIOS (+ 3 v1 = 16 scenarios total)
    # =========================================================================
    # 1_remote_access_support
    add("1_remote_access_support", "english", 1, "T1|T2|T5", 4, True,
        f"{s['mfs']} Tech Support: To fix your failed transaction, install AnyDesk and share the 9-digit code plus your PIN.")
    add("1_remote_access_support", "bangla", 1, "T1|T2|T5", 4, True,
        f"{s['bank']} সাপোর্ট: আপনার অ্যাপ ঠিক করতে টিমভিউয়ার ইনস্টল করে কোড ও পিন দিন।")
    add("1_remote_access_support", "banglish", 1, "T1|T2|T5", 4, True,
        f"{s['mfs']} helpline: Screen share app download korun ar code ta bolen, na hole taka ferot paben na.")
    n_t, mt = _wrap_multiturn(rng, s, [
        f"Sir, ami {s['mfs']} technical team theke bolchi. Apnar app e bug hoyeche.",
        "Ki bug? Amar balance to thik ache.",
        "Security issue ache. Play Store theke AnyDesk install kore code ta den ar PIN verify korun."
    ], "greet")
    add("1_remote_access_support", "mixed", 1, "T1|T2|T5", 4, True, mt, n_t)

    # 1_wallet_limit_upgrade
    add("1_wallet_limit_upgrade", "english", 1, "T1|T2", 1, True,
        f"{s['mfs']} Notice: Upgrade your daily wallet limit to Tk 1,00,000. Reply with your PIN and current OTP.")
    add("1_wallet_limit_upgrade", "bangla", 1, "T1|T2", 1, True,
        f"{s['mfs']}: আপনার লেনদেনের সীমা বাড়াতে এখনই পিন ও এনআইডি নম্বর পাঠান।")
    add("1_wallet_limit_upgrade", "banglish", 1, "T1|T2|T5", 1, True,
        f"SABDHAN! {s['mfs']}: APNAR WALLET LIMIT BARANO HOBE. CARD NUMBER, EXPIRY AR PIN DEN.")
    add("1_wallet_limit_upgrade", "mixed", 1, "T1|T2", 1, True,
        f"VIP {s['mfs']} limit upgrade: verify with 6-digit OTP to unlock Tk 50,000 daily transfer.")

    # 1_sim_kyc_nid_photo
    add("1_sim_kyc_nid_photo", "english", 1, "T1|T2|T5", 1, True,
        f"{s['telco']} Alert: Your SIM registration is incomplete. Send NID front/back photo and MFS PIN within 2h or SIM stops.")
    add("1_sim_kyc_nid_photo", "bangla", 1, "T1|T2|T5", 1, True,
        f"{s['telco']} সতর্কতা: ২ ঘণ্টার মধ্যে এনআইডি ছবি ও {s['mfs']} পিন না দিলে সিম বন্ধ হয়ে যাবে।")
    add("1_sim_kyc_nid_photo", "banglish", 1, "T1|T2|T5", 1, True,
        f"Apnar SIM ar {s['mfs']} bondho hoye jabe. NID front back chobi ar OTP pathan ekhoni.")
    add("1_sim_kyc_nid_photo", "mixed", 1, "T1|T2|T5", 1, True,
        f"Biometric mismatch on {s['telco']}! Reply with NID number and {s['mfs']} PIN to avoid deactivation.")

    # 1_pwd_reset_link
    add("1_pwd_reset_link", "english", 1, "T1|T2|T5", 3, True,
        f"{s['bank']} Alert: Unauthorized password change detected! Cancel immediately by entering PIN at {s['purl']}")
    add("1_pwd_reset_link", "bangla", 1, "T1|T2|T5", 3, True,
        f"{s['mfs']} নিরাপত্তা সতর্কতা: আপনার পাসওয়ার্ড হ্যাক হওয়ার ঝুঁকিতে। পিন দিয়ে যাচাই করুন: {s['purl']}")
    add("1_pwd_reset_link", "banglish", 1, "T1|T2|T5", 3, True,
        f"Apnar {s['bank']} password reset korte {s['purl']} e dhuke old PIN ar OTP din.")
    add("1_pwd_reset_link", "mixed", 1, "T1|T2", 3, True,
        f"{s['mfs']} web login: confirm your PIN and OTP on {s['purl']} to restore access.")

    # 1_agent_commission_fraud
    add("1_agent_commission_fraud", "english", 1, "T1|T2|T4", 1, True,
        f"{s['mfs']} Agent Desk: Claim Tk {s['amt']} monthly agent commission by sharing your merchant PIN and OTP.")
    add("1_agent_commission_fraud", "bangla", 1, "T1|T2|T4", 1, True,
        f"{s['mfs']} এজেন্ট কমিশন: {s['amt']} টাকা বোনাস পেতে আপনার এজেন্ট পিন ও ওটিপি দিন।")
    add("1_agent_commission_fraud", "banglish", 1, "T1|T2|T4", 1, True,
        f"Agent bhai, apnar {s['mfs']} commission {s['amt']} taka add korte OTP ta bolen.")
    add("1_agent_commission_fraud", "mixed", 1, "T1|T2|T4", 1, True,
        f"Merchant cashback Tk {s['amt']} ready for {s['mfs']}! Enter PIN at {s['purl']}.")

    # 1_reply_code_alert
    add("1_reply_code_alert", "english", 1, "T1|T2|T5", 1, True,
        f"URGENT {s['mfs']}: Reply to this SMS with the 6-digit verification code you received to stop a Tk {s['amt']} debit.")
    add("1_reply_code_alert", "bangla", 1, "T1|T2|T5", 1, True,
        f"জরুরি: আপনার {s['mfs']} থেকে {s['amt']} টাকা কাটা ঠেকাতে এখনই আসা ৬ সংখ্যার কোডটি রিপ্লাই করুন।")
    add("1_reply_code_alert", "banglish", 1, "T1|T2|T5", 1, True,
        f"Apnar number e ekta 6 digit code gese. Account safe rakhte code ta reply korun ekhoni.")
    add("1_reply_code_alert", "mixed", 1, "T1|T2|T5", 1, True,
        f"Security code {s['otp']}? Reply the code here to verify your {s['bank']} login.")

    # 1_dormant_reactivation
    add("1_dormant_reactivation", "english", 1, "T1|T2|T5", 1, True,
        f"{s['bank']}: Your account is marked dormant. Reactivate by sending A/C number, ATM PIN, and OTP.")
    add("1_dormant_reactivation", "bangla", 1, "T1|T2|T5", 1, True,
        f"{s['bank']}: আপনার অ্যাকাউন্ট নিষ্ক্রিয় হয়েছে। চালু করতে এটিএম পিন ও ওটিপি পাঠান।")
    add("1_dormant_reactivation", "banglish", 1, "T1|T2|T5", 1, True,
        f"Apnar {s['mfs']} account dormant hoye gese. Chalu korte PIN ar NID number pathan.")
    add("1_dormant_reactivation", "mixed", 1, "T1|T2", 1, True,
        f"Reactivate {s['mfs']} wallet: share last 4 digits of NID and your 5-digit PIN.")

    # 1_card_unblock_otp
    add("1_card_unblock_otp", "english", 1, "T1|T2|T5", 1, True,
        f"{s['bank']} Card Dept: Your debit card is temporarily blocked. Share OTP and CVV to unblock.")
    add("1_card_unblock_otp", "bangla", 1, "T1|T2|T5", 1, True,
        f"{s['bank']}: আপনার ডেবিট কার্ড ব্লক করা হয়েছে। আনব্লক করতে সিভিভি ও ওটিপি দিন।")
    add("1_card_unblock_otp", "banglish", 1, "T1|T2|T5", 1, True,
        f"Visa card block hoyeche. Unblock korte card expiry ar OTP ta bolen.")
    add("1_card_unblock_otp", "mixed", 1, "T1|T2|T5", 1, True,
        f"{s['bank']} card alert: unblock within 30 min by replying OTP.")

    # 1_govt_subsidy_pin
    add("1_govt_subsidy_pin", "english", 1, "T1|T2|T4", 1, False,
        f"Govt Relief Fund: Receive Tk {s['amt']} directly in {s['mfs']}. Share your {s['mfs']} PIN and OTP to verify wallet.")
    add("1_govt_subsidy_pin", "bangla", 1, "T1|T2|T4", 1, False,
        f"সরকারি ভাতা {s['amt']} টাকা পেতে আপনার {s['mfs']} পিন ও ওটিপি নম্বর পাঠান।")
    add("1_govt_subsidy_pin", "banglish", 1, "T1|T2|T4", 1, False,
        f"Shikkha britti {s['amt']} taka {s['mfs']} e nite apnar PIN ar OTP din.")
    add("1_govt_subsidy_pin", "mixed", 1, "T1|T2|T4", 1, False,
        f"Krishok/Student allowance Tk {s['amt']}: verify {s['mfs']} by sending PIN.")

    # 1_merchant_verify_pin
    add("1_merchant_verify_pin", "english", 1, "T1|T2", 1, False,
        f"{s['ecom']} Seller Center: Verify payout wallet by entering your {s['mfs']} PIN and OTP.")
    add("1_merchant_verify_pin", "banglish", 1, "T1|T2", 1, False,
        f"{s['ecom']} vendor payment {s['amt']} taka nite {s['mfs']} PIN verify korun.")
    add("1_merchant_verify_pin", "bangla", 1, "T1|T2", 1, False,
        f"{s['ecom']} পেমেন্ট রিসিভ করতে আপনার {s['mfs']} পিন ও ওটিপি দিন।")
    add("1_merchant_verify_pin", "mixed", 1, "T1|T2", 1, False,
        f"Payout hold on {s['ecom']}: share {s['mfs']} OTP to release Tk {s['amt']}.")

    # 1_fake_support_call
    n_t, mt = _wrap_multiturn(rng, s, [
        f"Hello, {s['mfs']} head office theke bolchi. Apnar account e ekta complain asche.",
        "Ki complain?",
        "Keu apnar account hack korar chesta korche. Ami ekta code pathacchi, oita bolle block kore dibo."
    ], "greet")
    add("1_fake_support_call", "banglish", 1, "T1|T2|T5", 1, True, mt, n_t)
    add("1_fake_support_call", "english", 1, "T1|T2|T5", 5, True,
        f"{s['bank']} Fraud Dept: Call {s['phone']} immediately and provide your PIN to stop unauthorized login.")
    add("1_fake_support_call", "bangla", 1, "T1|T2|T5", 1, True,
        f"{s['mfs']} হেল্পডেস্ক: হ্যাকিং ঠেকাতে এখনই আপনার পিন ও ওটিপি বলুন।")
    add("1_fake_support_call", "mixed", 1, "T1|T2|T5", 1, True,
        f"{s['bank']} support: share the 6-digit security PIN sent to your phone.")

    # 1_biometric_sync_otp
    add("1_biometric_sync_otp", "english", 1, "T1|T2", 1, False,
        f"{s['mfs']} App Update: Sync biometric login by replying with your current PIN and OTP.")
    add("1_biometric_sync_otp", "bangla", 1, "T1|T2", 1, False,
        f"{s['mfs']} ফিঙ্গারপ্রিন্ট চালু করতে পিন ও ওটিপি রিপ্লাই করুন।")
    add("1_biometric_sync_otp", "banglish", 1, "T1|T2", 1, False,
        f"Face ID / Fingerprint update korte {s['mfs']} PIN ta pathan.")
    add("1_biometric_sync_otp", "mixed", 1, "T1|T2", 1, False,
        f"Biometric sync failed on {s['bank']}. Send OTP to complete.")

    # 1_security_breach_harvest
    add("1_security_breach_harvest", "english", 1, "T1|T2|T5", 1, True,
        f"ALERT: {s['mfs']} database breach! Secure your Tk {s['bal']} balance by sending your PIN and password now.")
    add("1_security_breach_harvest", "bangla", 1, "T1|T2|T5", 1, True,
        f"সতর্কতা: আপনার {s['mfs']} ব্যালেন্স সুরক্ষিত রাখতে এখনই পিন ও পাসওয়ার্ড পাঠান।")
    add("1_security_breach_harvest", "banglish", 1, "T1|T2|T5", 1, True,
        f"Server hack hoyeche! Taka bachate {s['mfs']} PIN ar OTP ekhoni pathan.")
    add("1_security_breach_harvest", "mixed", 1, "T1|T2|T5", 1, True,
        f"Protect your {s['bank']} funds: reply with card PIN immediately.")

    # =========================================================================
    # CLASS 2: 11 NEW SCENARIOS (+ 4 v1 = 15 scenarios total)
    # =========================================================================
    # 2_eid_gift_scam
    add("2_eid_gift_scam", "english", 2, "T1|T4|T5", 3, True,
        f"Eid Mubarak! {s['mfs']} is gifting Tk {s['amt']} Eidi to lucky users. Claim before midnight: {s['purl']}")
    add("2_eid_gift_scam", "bangla", 2, "T1|T4|T5", 3, True,
        f"ঈদ মোবারক! {s['telco']} থেকে {s['amt']} টাকা ঈদ উপহার জিতেছেন। নিতে ক্লিক করুন: {s['purl']}")
    add("2_eid_gift_scam", "banglish", 2, "T1|T4", 3, True,
        f"Eid gift! {s['ecom']} theke free gift box paben. Link e click kore claim korun: {s['purl']}")
    add("2_eid_gift_scam", "mixed", 2, "T4|T3", 2, True,
        f"Eid Salami Tk {s['amt']} won! Pay Tk 50 verification charge via {s['mfs']} to receive.")

    # 2_telco_bundle_gift
    add("2_telco_bundle_gift", "english", 2, "T1|T4|T5", 3, True,
        f"{s['telco']} Free 25GB Internet Gift! Claim your free data pack now at {s['purl']}.")
    add("2_telco_bundle_gift", "bangla", 2, "T1|T4|T5", 3, True,
        f"{s['telco']} ফ্রি ২০ জিবি ইন্টারনেট গিফট! অফারটি পেতে ১ ঘণ্টার মধ্যে ক্লিক করুন: {s['purl']}")
    add("2_telco_bundle_gift", "banglish", 2, "T1|T4", 3, True,
        f"Apnar {s['telco']} sim e free 15GB data gift asche! Claim korte {s['purl']} e jan.")
    add("2_telco_bundle_gift", "mixed", 2, "T1|T4|T5", 3, True,
        f"Congrats! {s['telco']} 500 min + 10GB free reward selected for you. Tap {s['purl']}.")

    # 2_survey_reward
    add("2_survey_reward", "english", 2, "T1|T4", 3, False,
        f"Answer 3 quick questions about {s['ecom']} and win a guaranteed Tk {s['amt']} gift card: {s['purl']}")
    add("2_survey_reward", "bangla", 2, "T1|T4", 3, False,
        f"{s['ecom']} সম্পর্কে ৩টি প্রশ্নের উত্তর দিয়ে জিতে নিন {s['amt']} টাকা পুরস্কার: {s['purl']}")
    add("2_survey_reward", "banglish", 2, "T4", 3, False,
        f"Choto ekta survey fill korun ar peye jan Tk {s['amt']} voucher: {s['purl']}")
    add("2_survey_reward", "mixed", 2, "T1|T4", 3, False,
        f"{s['mfs']} user feedback reward: answer 4 questions to win Tk {s['amt']} at {s['purl']}.")

    # 2_govt_grant_scam
    add("2_govt_grant_scam", "english", 2, "T1|T3|T4", 2, False,
        f"National Digital Grant: You are awarded Tk {s['amt']} cash support! Pay Tk 200 processing fee to claim.")
    add("2_govt_grant_scam", "bangla", 2, "T1|T4|T5", 3, False,
        f"সরকারি বিশেষ অনুদান {s['amt']} টাকা আপনার নামে এসেছে! পেতে এখনই আবেদন করুন: {s['purl']}")
    add("2_govt_grant_scam", "banglish", 2, "T1|T3|T4", 2, False,
        f"Govt grant Tk {s['amt']} pitechen! Fee Tk 150 {s['mfs']} kore claim korun {s['phone']} e.")
    add("2_govt_grant_scam", "mixed", 2, "T1|T4", 3, False,
        f"Youth Development Cash Award Tk {s['amt']} selected for your number! Claim: {s['purl']}")

    # 2_anniversary_reward
    add("2_anniversary_reward", "english", 2, "T1|T4|T5", 3, True,
        f"{s['ecom']} 10th Anniversary Celebration! You won a Tk {s['amt']} shopping voucher. Claim: {s['purl']}")
    add("2_anniversary_reward", "bangla", 2, "T1|T4", 3, True,
        f"{s['mfs']} বর্ষপূর্তি উপহার! আপনি জিতেছেন {s['amt']} টাকা বোনাস। লিংকে যান: {s['purl']}")
    add("2_anniversary_reward", "banglish", 2, "T1|T4", 3, True,
        f"{s['telco']} birthday gift: apni Tk {s['amt']} jitechen! Click {s['purl']}.")
    add("2_anniversary_reward", "mixed", 2, "T1|T4|T5", 3, True,
        f"Happy Anniversary from {s['bank']}! Claim your free Tk {s['amt']} reward today: {s['purl']}")

    # 2_scratch_card_win
    add("2_scratch_card_win", "english", 2, "T4|T5", 3, False,
        f"You received a digital scratch card worth up to Tk {s['amt']}! Scratch & claim within 2 hours: {s['purl']}")
    add("2_scratch_card_win", "bangla", 2, "T4", 3, False,
        f"আপনি একটি স্ক্র্যাচ কার্ড জিতেছেন ({s['amt']} টাকা পর্যন্ত)! সংগ্রহ করুন: {s['purl']}")
    add("2_scratch_card_win", "banglish", 2, "T4|T3", 2, False,
        f"Scratch card e laptop jitechen! Delivery charge Tk 350 pay kore claim korun.")
    add("2_scratch_card_win", "mixed", 2, "T4|T5", 4, False,
        f"Scratch & Win! Download reward app from {s['purl']}/win.apk to unlock Tk {s['amt']}.")

    # 2_referral_bonus_scam
    add("2_referral_bonus_scam", "english", 2, "T4|T5", 3, True,
        f"Congrats! Your number won the Tk {s['amt']} lucky referral bonus! Click {s['purl']} before expiry.")
    add("2_referral_bonus_scam", "bangla", 2, "T4", 3, True,
        f"অভিনন্দন! রেফারেল লটারিতে আপনি {s['amt']} টাকা জিতেছেন। লিংকে ক্লিক করে নিন: {s['purl']}")
    add("2_referral_bonus_scam", "banglish", 2, "T4|T3", 2, True,
        f"Bonus Tk {s['amt']} jitechen! Transfer fee Tk 100 pathan {s['phone']} e.")
    add("2_referral_bonus_scam", "mixed", 2, "T4", 5, True,
        f"Referral Jackpot Tk {s['amt']} won! WhatsApp {s['phone']} to receive prize.")

    # 2_quiz_win_prize
    n_t, mt = _wrap_multiturn(rng, s, [
        f"Congratulations {s['name']}! Your number won our TV Quiz mega prize!",
        "Ami to kono quiz kheli nai.",
        f"Random mobile draw te select hoyechen. Tk {s['amt']} jitechen! Claim korte {s['purl']} e click korun."
    ], "greet")
    add("2_quiz_win_prize", "banglish", 2, "T4|T5", 3, False, mt, n_t)
    add("2_quiz_win_prize", "english", 2, "T3|T4", 2, False,
        f"Quiz Winner! You won an iPad Pro. Pay Tk {s['amt']} tax/registration fee to claim.")
    add("2_quiz_win_prize", "bangla", 2, "T4", 5, False,
        f"কুইজ বিজয়ী! আপনি স্মার্টটিভি জিতেছেন। নিতে {s['phone']} নম্বরে কল করুন।")
    add("2_quiz_win_prize", "mixed", 2, "T4", 1, False,
        f"TV Show Lucky Winner: reply with your full name, address & NID to claim Tk {s['amt']}.")

    # 2_festival_raffle
    add("2_festival_raffle", "english", 2, "T4|T3", 2, False,
        f"Pohela Boishakh Mega Raffle: You won Tk {s['amt']}! Pay Tk 200 stamp fee to release prize.")
    add("2_festival_raffle", "bangla", 2, "T4|T5", 3, False,
        f"বৈশাখী মেগা র‍্যাফেল ড্র-তে আপনি {s['amt']} টাকা জিতেছেন! এখনই ক্লেইম করুন: {s['purl']}")
    add("2_festival_raffle", "banglish", 2, "T4", 5, False,
        f"New Year raffle winner! Gold coin jitechen, WhatsApp korun {s['phone']}.")
    add("2_festival_raffle", "mixed", 2, "T1|T4", 3, False,
        f"{s['ecom']} Winter Raffle: Tk {s['amt']} gift card won! Claim at {s['purl']}.")

    # 2_vip_cash_prize
    add("2_vip_cash_prize", "english", 2, "T1|T4|T5", 3, False,
        f"VIP Member Reward: {s['mfs']} selected you for Tk {s['amt']} cash prize. Claim within 30 mins: {s['purl']}")
    add("2_vip_cash_prize", "bangla", 2, "T1|T4", 2, False,
        f"ভিআইপি পুরস্কার: {s['mfs']} থেকে {s['amt']} টাকা জিতেছেন। প্রসেসিং ফি ২০০ টাকা পাঠান।")
    add("2_vip_cash_prize", "banglish", 2, "T1|T4", 5, False,
        f"VIP customer draw te apni Tk {s['amt']} jitechen! Call {s['phone']} to claim.")
    add("2_vip_cash_prize", "mixed", 2, "T4", 1, False,
        f"VIP Prize Tk {s['amt']}: reply with your {s['mfs']} number and address to receive.")

    # 2_foreign_lottery
    add("2_foreign_lottery", "english", 2, "T3|T4|T5", 2, False,
        f"UK International Lottery: Your email/number won $10,000! Pay Tk {s['amt']} transfer clearance fee.")
    add("2_foreign_lottery", "bangla", 2, "T3|T4", 2, False,
        f"আন্তর্জাতিক লটারিতে আপনি ১০ লক্ষ টাকা জিতেছেন! ছাড়াতে {s['amt']} টাকা ফি দিন।")
    add("2_foreign_lottery", "banglish", 2, "T4", 5, False,
        f"Dubai lottery te apni Tk {s['amt']} jitechen! Claim korte {s['phone']} e WhatsApp korun.")
    add("2_foreign_lottery", "mixed", 2, "T4|T5", 3, False,
        f"Global Cash Prize won! Click {s['purl']} within 2h to claim your reward.")

    # =========================================================================
    # CLASS 3: 11 NEW SCENARIOS (+ 4 v1 = 15 scenarios total)
    # =========================================================================
    # 3_forex_bot
    add("3_forex_bot", "english", 3, "T5|T6", 4, False,
        f"Automated Forex AI Bot: Earn 40% daily profit passively! Join our Telegram channel: {s['purl']}")
    add("3_forex_bot", "bangla", 3, "T5|T6", 2, False,
        f"ফরেক্স ট্রেডিং বট: প্রতিদিন ৪০% নিশ্চিত লাভ! {s['amt']} টাকা ডিপোজিট করে শুরু করুন।")
    add("3_forex_bot", "banglish", 3, "T6", 4, False,
        f"Forex signal bot diye daily Tk {s['amt']} income korun. Join VIP: {s['purl']}")
    add("3_forex_bot", "mixed", 3, "T3|T6", 2, False,
        f"Auto Forex Trading: deposit Tk {s['amt']}, guaranteed weekly return 200%.")

    # 3_mlm_network
    add("3_mlm_network", "english", 3, "T3|T6", 2, False,
        f"Build your financial freedom! Join our network marketing team with Tk {s['amt']} starter kit and earn lakhs monthly.")
    add("3_mlm_network", "bangla", 3, "T3|T6", 4, False,
        f"নেটওয়ার্ক মার্কেটিংয়ে যোগ দিয়ে মাসে লক্ষ টাকা আয় করুন! জয়েন করতে লিংকে যান: {s['purl']}")
    add("3_mlm_network", "banglish", 3, "T6", 4, False,
        f"Matro 3 jon member join koralei Tk {s['amt']} bonus! Aj e group e join korun.")
    add("3_mlm_network", "mixed", 3, "T3|T6", 2, False,
        f"Diamond membership fee Tk {s['amt']}: recruit 2 friends and earn passive income.")

    # 3_loan_advance_fee
    add("3_loan_advance_fee", "english", 3, "T3|T6", 2, True,
        f"Instant Personal Loan up to Tk 5,00,000 without documents! Pay Tk {s['amt']} processing fee first via {s['mfs']}.")
    add("3_loan_advance_fee", "bangla", 3, "T3|T6", 2, True,
        f"বিনা জামানতে ৫ লক্ষ টাকা লোন নিন! প্রসেসিং ফি {s['amt']} টাকা আগে পাঠান।")
    add("3_loan_advance_fee", "banglish", 3, "T3|T6", 2, True,
        f"Easy loan Tk 2 lakh 1 ghontay! Registration fee Tk {s['amt']} {s['mfs']} korun {s['phone']} e.")
    add("3_loan_advance_fee", "mixed", 3, "T3|T6", 5, True,
        f"Zero-interest instant loan approved! Call {s['phone']} and pay Tk {s['amt']} stamp charge.")

    # 3_page_admin_job
    add("3_page_admin_job", "english", 3, "T6", 4, False,
        f"Hiring Facebook Page Admins! Work 1 hour/day from phone, earn Tk {s['amt']} daily. Apply on WhatsApp: {s['phone']}")
    add("3_page_admin_job", "bangla", 3, "T6", 4, False,
        f"ফেসবুক পেজ অ্যাডমিন নিয়োগ চলছে! দিনে ১ ঘণ্টা কাজে {s['amt']} টাকা আয়। হোয়াটসঅ্যাপে যোগ দিন।")
    add("3_page_admin_job", "banglish", 3, "T6", 4, False,
        f"Page admin lagbe, daily salary Tk {s['amt']}. Telegram group e join korun: {s['purl']}")
    add("3_page_admin_job", "mixed", 3, "T3|T6", 2, False,
        f"E-commerce page moderator job: pay Tk {s['amt']} ID activation fee to start.")

    # 3_brushing_review_task
    add("3_brushing_review_task", "english", 3, "T1|T6", 4, False,
        f"Earn Tk {s['amt']} per 5-star review on {s['ecom']} products! Join our Telegram task group: {s['purl']}")
    add("3_brushing_review_task", "bangla", 3, "T6", 4, False,
        f"প্রোডাক্টে ফাইভ স্টার রিভিউ দিয়ে প্রতিদিন {s['amt']} টাকা আয় করুন। টেলিগ্রামে যোগ দিন।")
    add("3_brushing_review_task", "banglish", 3, "T6", 4, False,
        f"Google Maps e review dilei protiti review te Tk {s['amt']}! WhatsApp korun {s['phone']}.")
    add("3_brushing_review_task", "mixed", 3, "T3|T6", 2, False,
        f"Prepaid merchant order task: deposit Tk {s['amt']}, get 30% commission back in 10 mins.")

    # 3_app_testing_job
    add("3_app_testing_job", "english", 3, "T6", 4, False,
        f"Get paid Tk {s['amt']} for each app you download and test! Install our task app: {s['purl']}/task.apk")
    add("3_app_testing_job", "bangla", 3, "T6", 4, False,
        f"অ্যাপ ডাউনলোড ও টেস্ট করে প্রতিদিন {s['amt']} টাকা আয় করুন। অ্যাপ নামান: {s['purl']}")
    add("3_app_testing_job", "banglish", 3, "T6", 4, False,
        f"Notun app install kore review dile Tk {s['amt']} income! Join {s['purl']}.")
    add("3_app_testing_job", "mixed", 3, "T6", 4, False,
        f"Mobile app tester needed: earn $15/hour. Join Telegram: {s['purl']}")

    # 3_captcha_typing_job
    add("3_captcha_typing_job", "english", 3, "T3|T6", 2, False,
        f"Simple Captcha Typing Job: Earn Tk {s['amt']} daily! Pay Tk 500 software/ID fee to start.")
    add("3_captcha_typing_job", "bangla", 3, "T3|T6", 2, False,
        f"ক্যাপচা টাইপিং কাজ: দিনে {s['amt']} টাকা আয়। আইডি খুলতে ৩০০ টাকা ফি দিন।")
    add("3_captcha_typing_job", "banglish", 3, "T6", 4, False,
        f"Mobile diye captcha typing kore Tk {s['amt']} income. WhatsApp group e join korun.")
    add("3_captcha_typing_job", "mixed", 3, "T3|T6", 2, False,
        f"Form filling home job: guaranteed Tk 25,000/month. Registration fee Tk {s['amt']}.")

    # 3_hotel_booking_task
    n_t, mt = _wrap_multiturn(rng, s, [
        f"Hi, I am {s['name']} from {s['emp']} recruitment.",
        "Yes, what is the role?",
        f"We offer online hotel rating tasks. Earn Tk {s['amt']} per day in 20 minutes.",
        f"Join our Telegram mentor group to receive your first task: {s['purl']}"
    ], "casual")
    add("3_hotel_booking_task", "english", 3, "T1|T6", 4, False, mt, n_t)
    add("3_hotel_booking_task", "banglish", 3, "T6", 4, False,
        f"Hotel booking rating task kore daily Tk {s['amt']} income! Telegram e message din.")
    add("3_hotel_booking_task", "bangla", 3, "T3|T6", 2, False,
        f"অনলাইন হোটেল বুকিং টাস্ক: {s['amt']} টাকা রিচার্জ করে ৩০% কমিশন নিন।")
    add("3_hotel_booking_task", "mixed", 3, "T6", 4, False,
        f"Part-time resort rating job: earn Tk {s['amt']}/day. Join WhatsApp now.")

    # 3_telegram_vip_group
    add("3_telegram_vip_group", "english", 3, "T5|T6", 4, False,
        f"Secret Money-Making Telegram VIP Group! Limited 50 seats. Join now and earn Tk {s['amt']} today: {s['purl']}")
    add("3_telegram_vip_group", "bangla", 3, "T6", 4, False,
        f"টেলিগ্রাম ভিআইপি আর্নিং গ্রুপে যোগ দিন, প্রতিদিন {s['amt']} টাকা নিশ্চিত আয়: {s['purl']}")
    add("3_telegram_vip_group", "banglish", 3, "T6", 4, False,
        f"Daily earning Telegram channel e join korun, ajkei Tk {s['amt']} paben: {s['purl']}")
    add("3_telegram_vip_group", "mixed", 3, "T3|T6", 2, False,
        f"VIP Earning Club: pay Tk {s['amt']} entry fee for guaranteed daily payouts.")

    # 3_data_entry_job (hard negative twin of 0_interview_legit)
    add("3_data_entry_job", "english", 3, "T1|T3|T6", 2, True,
        f"HR, {s['emp']}: You are selected for remote Data Entry Officer! Pay Tk {s['amt']} training/laptop deposit to confirm.")
    add("3_data_entry_job", "bangla", 3, "T1|T3|T6", 2, True,
        f"{s['emp']} নিয়োগ: ঘরে বসে ডাটা এন্ট্রি চাকরির জন্য {s['amt']} টাকা জামানত পাঠান।")
    add("3_data_entry_job", "banglish", 3, "T1|T3|T6", 2, True,
        f"{s['emp']} e data entry job confirm korte Tk {s['amt']} registration fee {s['mfs']} korun.")
    add("3_data_entry_job", "mixed", 3, "T1|T3|T6", 5, True,
        f"Selected at {s['emp']}! Call {s['phone']} and pay Tk {s['amt']} ID card fee.")

    # 3_freelance_course_scam
    add("3_freelance_course_scam", "english", 3, "T3|T6", 2, True,
        f"100% Guaranteed $1,000/month Freelancing! Enroll in our secret course today for Tk {s['amt']}.")
    add("3_freelance_course_scam", "bangla", 3, "T3|T6", 2, True,
        f"১০০% নিশ্চিত মাসে ৫০,০০০ টাকা ফ্রিল্যান্সিং আয়! আজই {s['amt']} টাকা দিয়ে কোর্সে ভর্তি হোন।")
    add("3_freelance_course_scam", "banglish", 3, "T3|T6", 4, True,
        f"Dollar income course: matro {s['amt']} taka fee diye WhatsApp batch e join korun.")
    add("3_freelance_course_scam", "mixed", 3, "T3|T6", 2, True,
        f"Guaranteed Fiverr earning trick! Pay Tk {s['amt']} via {s['mfs']} to get access.")

    # =========================================================================
    # CLASS 4: 11 NEW SCENARIOS (+ 4 v1 = 15 scenarios total)
    # =========================================================================
    # 4_failed_delivery_redelivery
    add("4_failed_delivery_redelivery", "english", 4, "T1|T3|T5", 2, True,
        f"{s['courier']}: Delivery attempt #2 failed for {s['parcel']}. Pay Tk {s['amt']} redelivery fee at {s['purl']}")
    add("4_failed_delivery_redelivery", "bangla", 4, "T1|T3|T5", 2, True,
        f"{s['courier']}: আপনার পার্সেল {s['parcel']} ডেলিভারি করা যায়নি। পুনরায় পাঠাতে {s['amt']} টাকা ফি দিন: {s['purl']}")
    add("4_failed_delivery_redelivery", "banglish", 4, "T1|T3", 2, True,
        f"Delivery failed! Redelivery charge Tk {s['amt']} pay na korle parcel return hobe: {s['purl']}")
    add("4_failed_delivery_redelivery", "mixed", 4, "T1|T5", 3, True,
        f"{s['courier']} alert: reschedule delivery for #{s['parcel']} within 24h at {s['purl']}")

    # 4_cod_confirm_phish (hard negative twin of 0_cod_delivery_legit)
    add("4_cod_confirm_phish", "english", 4, "T1|T3", 2, True,
        f"{s['ecom']} Delivery: Your COD parcel #{s['parcel']} requires Tk {s['amt']} advance security deposit via {s['purl']} before dispatch.")
    add("4_cod_confirm_phish", "bangla", 4, "T1|T3", 2, True,
        f"আপনার সিওডি পার্সেল #{s['parcel']} পাঠানোর আগে {s['amt']} টাকা অগ্রিম পরিশোধ করুন: {s['purl']}")
    add("4_cod_confirm_phish", "banglish", 4, "T1|T3", 2, True,
        f"{s['courier']} COD parcel confirm korte age Tk {s['amt']} {s['mfs']} korun {s['phone']} e.")
    add("4_cod_confirm_phish", "mixed", 4, "T1|T3", 3, True,
        f"Confirm COD order #{s['parcel']} fee at {s['purl']} or order will be cancelled.")

    # 4_tracking_link_phish
    add("4_tracking_link_phish", "english", 4, "T1|T5", 3, True,
        f"{s['courier']}: Unusual tracking issue on parcel {s['parcel']}. Verify your delivery location immediately: {s['purl']}")
    add("4_tracking_link_phish", "bangla", 4, "T1|T5", 3, True,
        f"{s['courier']}: পার্সেল {s['parcel']} ট্র্যাকিং সমস্যা! লোকেশন নিশ্চিত করতে ক্লিক করুন: {s['purl']}")
    add("4_tracking_link_phish", "banglish", 4, "T1", 3, True,
        f"Apnar parcel {s['parcel']} track korte ar address confirm korte link e jan: {s['purl']}")
    add("4_tracking_link_phish", "mixed", 4, "T1|T5", 3, True,
        f"Live tracking paused for #{s['parcel']}. Update details at {s['purl']} today.")

    # 4_wrong_parcel_claim
    n_t, mt = _wrap_multiturn(rng, s, [
        f"Hello, this is {s['courier']} hub. A high-value parcel #{s['parcel']} arrived under your phone number.",
        "I didn't order anything.",
        f"Return cancellation requires verification. Click {s['purl']} or pay Tk {s['amt']} return fee."
    ], "casual")
    add("4_wrong_parcel_claim", "english", 4, "T1|T3|T5", 2, True, mt, n_t)
    add("4_wrong_parcel_claim", "banglish", 4, "T1|T3", 2, True,
        f"Apnar name e ekta international parcel asche. Cancel korte holeo Tk {s['amt']} charge dite hobe.")
    add("4_wrong_parcel_claim", "bangla", 4, "T1|T5", 3, True,
        f"ভুল ঠিকানার পার্সেল #{s['parcel']} বাতিল করতে এই লিংকে যান: {s['purl']}")
    add("4_wrong_parcel_claim", "mixed", 4, "T1|T3", 5, True,
        f"Unclaimed parcel #{s['parcel']} penalty! Call {s['phone']} to avoid fine.")

    # 4_parcel_vat_tax
    add("4_parcel_vat_tax", "english", 4, "T1|T3|T5", 2, False,
        f"Airport Parcel Clearance: Your package #{s['parcel']} owes Tk {s['amt']} VAT/Tax. Pay via {s['mfs']} {s['phone']} today.")
    add("4_parcel_vat_tax", "bangla", 4, "T1|T3|T5", 2, False,
        f"বিমানবন্দর কাস্টমস: আপনার পার্সেলে {s['amt']} টাকা ভ্যাট বকেয়া আছে। আজই পরিশোধ করুন।")
    add("4_parcel_vat_tax", "banglish", 4, "T1|T3", 2, False,
        f"Foreign parcel e Tk {s['amt']} tax baki ache. Clear korte {s['purl']} e pay korun.")
    add("4_parcel_vat_tax", "mixed", 4, "T1|T3|T5", 2, False,
        f"Customs VAT notice #{s['parcel']}: pay Tk {s['amt']} or parcel will be auctioned.")

    # 4_courier_kyc_verify
    add("4_courier_kyc_verify", "english", 4, "T1|T2|T5", 3, True,
        f"{s['courier']}: Courier KYC required to release parcel #{s['parcel']}. Submit NID and OTP at {s['purl']}")
    add("4_courier_kyc_verify", "bangla", 4, "T1|T2", 3, True,
        f"{s['courier']}: পার্সেল পেতে কেওয়াইসি যাচাই করুন। এনআইডি ও কোড দিন: {s['purl']}")
    add("4_courier_kyc_verify", "banglish", 4, "T1|T2", 1, True,
        f"Parcel {s['parcel']} release korte apnar NID number ar phone e asha code ta reply korun.")
    add("4_courier_kyc_verify", "mixed", 4, "T1|T5", 3, True,
        f"{s['courier']} identity check: verify parcel recipient at {s['purl']}.")

    # 4_intl_gift_customs
    add("4_intl_gift_customs", "english", 4, "T3|T4|T5", 2, False,
        f"A friend abroad sent you a luxury gift box held at Dhaka customs! Pay Tk {s['amt']} clearance charge to {s['phone']}.")
    add("4_intl_gift_customs", "bangla", 4, "T3|T4", 2, False,
        f"বিদেশ থেকে আপনার জন্য পাঠানো গিফট পার্সেল কাস্টমসে আটকে আছে। ছাড়াতে {s['amt']} টাকা পাঠান।")
    add("4_intl_gift_customs", "banglish", 4, "T3|T4", 2, False,
        f"London theke gift parcel asche, customs clearance Tk {s['amt']} {s['mfs']} korun.")
    add("4_intl_gift_customs", "mixed", 4, "T1|T3", 5, False,
        f"Diplomatic gift parcel #{s['parcel']} held at airport. Call agent {s['phone']}.")

    # 4_damaged_parcel_comp
    add("4_damaged_parcel_comp", "english", 4, "T1|T5", 3, False,
        f"{s['courier']}: Your parcel #{s['parcel']} was damaged in transit. Claim Tk {s['amt']} compensation at {s['purl']}")
    add("4_damaged_parcel_comp", "bangla", 4, "T1|T5", 3, False,
        f"{s['courier']}: আপনার পার্সেল ক্ষতিগ্রস্ত হয়েছে। ক্ষতিপূরণ পেতে ক্লিক করুন: {s['purl']}")
    add("4_damaged_parcel_comp", "banglish", 4, "T1", 3, False,
        f"Parcel damage compensation Tk {s['amt']} nite {s['purl']} e form fill korun.")
    add("4_damaged_parcel_comp", "mixed", 4, "T1|T3", 2, False,
        f"Claim parcel insurance Tk {s['amt']}: pay Tk 100 verification fee at {s['purl']}.")

    # 4_hub_pickup_fee
    add("4_hub_pickup_fee", "english", 4, "T1|T3", 2, True,
        f"{s['courier']}: Parcel {s['parcel']} is at local hub. Pay Tk {s['amt']} hub handling fee before pickup: {s['purl']}")
    add("4_hub_pickup_fee", "bangla", 4, "T1|T3", 2, True,
        f"{s['courier']}: হাব থেকে পার্সেল #{s['parcel']} তুলতে {s['amt']} টাকা সার্ভিস চার্জ দিন: {s['purl']}")
    add("4_hub_pickup_fee", "banglish", 4, "T1|T3", 2, True,
        f"Hub e parcel asche, Tk {s['amt']} gate pass fee {s['mfs']} kore collect korun.")
    add("4_hub_pickup_fee", "mixed", 4, "T1|T3", 3, True,
        f"Pay Tk {s['amt']} locker fee to unlock parcel #{s['parcel']} at {s['purl']}.")

    # 4_warehouse_penalty
    add("4_warehouse_penalty", "english", 4, "T1|T3|T5", 2, True,
        f"FINAL NOTICE: Parcel #{s['parcel']} incurring Tk {s['amt']}/day warehouse penalty! Pay now at {s['purl']}")
    add("4_warehouse_penalty", "bangla", 4, "T1|T3|T5", 2, True,
        f"শেষ সতর্কতা: পার্সেল #{s['parcel']} গুদামে রাখায় প্রতিদিন {s['amt']} টাকা জরিমানা হচ্ছে। এখনই পরিশোধ করুন।")
    add("4_warehouse_penalty", "banglish", 4, "T1|T3|T5", 2, True,
        f"Warehouse charge Tk {s['amt']} hoye gese! Ajke pay na korle legal action: {s['purl']}")
    add("4_warehouse_penalty", "mixed", 4, "T1|T5", 5, True,
        f"Overdue parcel storage #{s['parcel']}: call {s['phone']} immediately.")

    # 4_courier_pin_confirm
    add("4_courier_pin_confirm", "english", 4, "T1|T2", 1, False,
        f"{s['courier']}: To confirm delivery of parcel #{s['parcel']}, reply with the 6-digit OTP sent to your phone.")
    add("4_courier_pin_confirm", "bangla", 4, "T1|T2", 1, False,
        f"{s['courier']}: পার্সেল #{s['parcel']} গ্রহণ নিশ্চিত করতে ফোনে আসা ওটিপি কোডটি বলুন।")
    add("4_courier_pin_confirm", "banglish", 4, "T1|T2", 1, False,
        f"Parcel release korte apnar phone e asha verification code ta reply korun.")
    add("4_courier_pin_confirm", "mixed", 4, "T1|T2", 3, False,
        f"Confirm delivery #{s['parcel']} by entering OTP at {s['purl']}.")

    # =========================================================================
    # CLASS 5: 16 NEW SCENARIOS (+ 4 v1 = 20 scenarios total)
    # =========================================================================
    # 5_restaurant_promo
    add("5_restaurant_promo", "english", 5, "NONE", 0, False,
        f"Pizza Hut Weekend Combo: 2 large pizzas + garlic bread for Tk {s['amt']} today only! Order in app.")
    add("5_restaurant_promo", "bangla", 5, "NONE", 0, False,
        f"বিরিয়ানি উৎসব! আজ অর্ডার করলেই পাচ্ছেন ২০% ডিসকাউন্ট, কোড: FOOD20।")
    add("5_restaurant_promo", "banglish", 5, "NONE", 0, False,
        f"Burger Lover! Buy 1 get 1 free on all combos till 10 PM at Dhanmondi outlet.")
    add("5_restaurant_promo", "mixed", 5, "NONE", 0, False,
        f"KFC Wednesday deal: 8 pcs hot & crispy Tk {s['amt']}. Visit nearest branch.")

    # 5_telco_bundles
    add("5_telco_bundles", "english", 5, "NONE", 0, True,
        f"{s['telco']}: Special 10GB + 200 min bundle for Tk {s['amt']}, valid 7 days! Dial *121*5# to buy.")
    add("5_telco_bundles", "bangla", 5, "NONE", 0, True,
        f"{s['telco']} দারুণ অফার! {s['amt']} টাকায় ১৫ জিবি ইন্টারনেট ৭ দিন মেয়াদে। ডায়াল *১২১#।")
    add("5_telco_bundles", "banglish", 5, "NONE", 0, True,
        f"{s['telco']} combo pack: 5GB + 100 min matro {s['amt']} taka! MyApp theke kinun.")
    add("5_telco_bundles", "mixed", 5, "NONE", 0, True,
        f"{s['telco']} night pack 20GB at Tk {s['amt']}. Dial *5000# to activate.")

    # 5_flash_sale
    add("5_flash_sale", "english", 5, "NONE", 0, True,
        f"{s['ecom']} 11.11 Flash Sale starts at midnight! Discounts up to 60% + free shipping. Shop: {s['lurl']}")
    add("5_flash_sale", "bangla", 5, "NONE", 0, True,
        f"{s['ecom']} ফ্ল্যাশ সেল! আজ রাত ১২টা থেকে সব পণ্যে ৫০% পর্যন্ত ছাড়: {s['lurl']}")
    add("5_flash_sale", "banglish", 5, "NONE", 0, True,
        f"Mega Flash Sale! Ajke rat 12 tay {s['ecom']} e 40% off. Voucher code FLASH40.")
    add("5_flash_sale", "mixed", 5, "NONE", 0, True,
        f"Limited time flash deal at {s['ecom']}! Save Tk {s['amt']} on electronics.")

    # 5_insurance_offer
    add("5_insurance_offer", "english", 5, "NONE", 0, False,
        f"{s['bank']} Life Care: Secure your family's future with health insurance starting Tk {s['amt']}/month. Call {s['phone']}.")
    add("5_insurance_offer", "bangla", 5, "NONE", 0, False,
        f"স্বাস্থ্য সুরক্ষা বীমা: মাসে মাত্র {s['amt']} টাকায় পরিবারের চিকিৎসা কভারেজ নিন।")
    add("5_insurance_offer", "banglish", 5, "NONE", 0, False,
        f"Family health insurance plan, bochore 5 lakh cover. Details jante {s['lurl']} dekhun.")
    add("5_insurance_offer", "mixed", 5, "NONE", 0, False,
        f"Protect your car with {s['bank']} auto insurance. 15% early renewal discount.")

    # 5_bank_loan_offer (hard negative twin of 3_loan_advance_fee)
    add("5_bank_loan_offer", "english", 5, "NONE", 0, True,
        f"{s['bank']}: Pre-approved personal loan at 9.5% interest! Apply at your nearest branch or {s['lurl']}. T&C apply.")
    add("5_bank_loan_offer", "bangla", 5, "NONE", 0, True,
        f"{s['bank']} হোম লোন অফার: সহজ কিস্তিতে বাড়ি বা ফ্ল্যাট কিনুন। বিস্তারিত জানতে শাখায় যোগাযোগ করুন।")
    add("5_bank_loan_offer", "banglish", 5, "NONE", 0, True,
        f"{s['bank']} credit card ar auto loan offer! 0% processing fee this month. Apply in app.")
    add("5_bank_loan_offer", "mixed", 5, "NONE", 0, True,
        f"SME business loan from {s['bank']}. Visit {s['lurl']} for eligibility criteria.")

    # 5_event_invitation
    add("5_event_invitation", "english", 5, "NONE", 0, False,
        f"You're invited to Dhaka Tech Expo 2026 at ICCB! Register your free visitor pass at {s['lurl']}.")
    add("5_event_invitation", "bangla", 5, "NONE", 0, False,
        f"ঢাকা বইমেলা ও সাহিত্য উৎসবে আপনাকে আমন্ত্রণ! স্টল নম্বর ৪৫ ভিজিট করুন।")
    add("5_event_invitation", "banglish", 5, "NONE", 0, False,
        f"Free Career Seminar this Saturday at Banani. Seat book korte {s['lurl']} e jan.")
    add("5_event_invitation", "mixed", 5, "NONE", 0, False,
        f"Live Music Fest tickets now available! Early bird 20% discount with code MUSIC20.")

    # 5_gym_fitness
    add("5_gym_fitness", "english", 5, "NONE", 0, False,
        f"FitZone Gym Gulshan: 3-month membership at Tk {s['amt']}! Free trainer assessment this week.")
    add("5_gym_fitness", "bangla", 5, "NONE", 0, False,
        f"ফিটনেস ক্লাবে ভর্তি চলছে! প্রথম মাসের ফি-তে ৩০% ছাড়।")
    add("5_gym_fitness", "banglish", 5, "NONE", 0, False,
        f"Summer fitness offer: 6 month gym package matro Tk {s['amt']}. Call {s['phone']}.")
    add("5_gym_fitness", "mixed", 5, "NONE", 0, False,
        f"Yoga & Zumba morning batch open! Free trial class Friday.")

    # 5_travel_deal
    add("5_travel_deal", "english", 5, "NONE", 0, False,
        f"Cox's Bazar Weekend Package: 2 nights hotel + flight starting Tk {s['amt']}! Book at {s['lurl']}.")
    add("5_travel_deal", "bangla", 5, "NONE", 0, False,
        f"সাজেক ও কক্সবাজার ভ্রমণ প্যাকেজ! গ্রুপ বুকিংয়ে ১৫% ছাড়। কল করুন {s['phone']}।")
    add("5_travel_deal", "banglish", 5, "NONE", 0, False,
        f"Sylhet resort booking e 25% discount! Use promo SYLHET25.")
    add("5_travel_deal", "mixed", 5, "NONE", 0, False,
        f"Domestic flight tickets starting Tk {s['amt']}. Book early via official app.")

    # 5_electronics_sale
    add("5_electronics_sale", "english", 5, "NONE", 0, True,
        f"Mega Electronics Clearance: Up to 35% off smartphones & laptops at {s['ecom']} showroom! T&C apply.")
    add("5_electronics_sale", "bangla", 5, "NONE", 0, True,
        f"শীতকালীন ইলেকট্রনিক্স মেলা! এসি ও ফ্রিজে ২০% ডিসকাউন্ট ও ফ্রি হোম ডেলিভারি।")
    add("5_electronics_sale", "banglish", 5, "NONE", 0, True,
        f"Smart TV kinle soundbar free! Offer cholbe ei soptaho porjonto.")
    add("5_electronics_sale", "mixed", 5, "NONE", 0, True,
        f"0% EMI on laptops for 12 months with {s['bank']} cards at {s['ecom']}.")

    # 5_fashion_collection
    add("5_fashion_collection", "english", 5, "NONE", 0, False,
        f"{s['ecom']} New Winter Collection is here! Flat 25% off jackets and hoodies in-store and online.")
    add("5_fashion_collection", "bangla", 5, "NONE", 0, False,
        f"নতুন ঈদ কালেকশন এখন সব শোরুমে! পাঞ্জাবি ও শাড়িতে আকর্ষণীয় মূল্যছাড়।")
    add("5_fashion_collection", "banglish", 5, "NONE", 0, False,
        f"End of season fashion sale! Sob panjabi ar kurti te 40% off.")
    add("5_fashion_collection", "mixed", 5, "NONE", 0, False,
        f"Buy 2 get 1 free on formal shirts at {s['ecom']} outlet.")

    # 5_grocery_promo
    add("5_grocery_promo", "english", 5, "NONE", 0, False,
        f"{s['ecom']} Grocery: Free delivery on orders above Tk 999! Fresh vegetables & fruits delivered in 1 hour.")
    add("5_grocery_promo", "bangla", 5, "NONE", 0, False,
        f"তাজা বাজার ও গ্রোসারি অর্ডারে ফ্রি ডেলিভারি! কোড ব্যবহার করুন: FRESH।")
    add("5_grocery_promo", "banglish", 5, "NONE", 0, False,
        f"Monthly bazar e Tk {s['amt']} instant discount! Order korun {s['ecom']} app theke.")
    add("5_grocery_promo", "mixed", 5, "NONE", 0, False,
        f"Weekend grocery deal: 10% off cooking oil and rice.")

    # 5_rideshare_promo
    add("5_rideshare_promo", "english", 5, "NONE", 0, False,
        f"Ride & Save! Get 30% off your next 3 bike or car rides with code RIDE30.")
    add("5_rideshare_promo", "bangla", 5, "NONE", 0, False,
        f"অফিস যাওয়ার রাইডে ৪০% ছাড়! আজই অ্যাপ থেকে রাইড বুক করুন।")
    add("5_rideshare_promo", "banglish", 5, "NONE", 0, False,
        f"Ajke sob bike ride e flat Tk 50 off! Promo code DHAKA50.")
    add("5_rideshare_promo", "mixed", 5, "NONE", 0, False,
        f"Zero surge pricing during afternoon hours! Book your ride now.")

    # 5_ott_streaming_deal
    add("5_ott_streaming_deal", "english", 5, "NONE", 0, False,
        f"Watch new Bangla web series & live cricket! Subscribe for Tk {s['amt']}/month at {s['lurl']}.")
    add("5_ott_streaming_deal", "bangla", 5, "NONE", 0, False,
        f"নতুন বাংলা সিনেমা ও লাইভ খেলা দেখুন! মাসিক প্যাক মাত্র {s['amt']} টাকা।")
    add("5_ott_streaming_deal", "banglish", 5, "NONE", 0, False,
        f"Annual OTT subscription e 50% off! Ajkei subscribe korun.")
    add("5_ott_streaming_deal", "mixed", 5, "NONE", 0, False,
        f"7-day free trial for premium movies and sports channels.")

    # 5_card_reward_promo
    add("5_card_reward_promo", "english", 5, "NONE", 0, True,
        f"{s['bank']} Cardholders: Earn 2x reward points on all dining & shopping spends this weekend. T&C apply.")
    add("5_card_reward_promo", "bangla", 5, "NONE", 0, True,
        f"{s['bank']} কার্ডে শপিং করলে ১০% ইনস্ট্যান্ট ক্যাশব্যাক! শর্ত প্রযোজ্য।")
    add("5_card_reward_promo", "banglish", 5, "NONE", 0, True,
        f"{s['mfs']} payment e 15% cashback paben {s['ecom']} e, max Tk 200.")
    add("5_card_reward_promo", "mixed", 5, "NONE", 0, True,
        f"Airport lounge access + 10% cashback with {s['bank']} Platinum Card.")

    # 5_health_checkup_pkg
    add("5_health_checkup_pkg", "english", 5, "NONE", 0, False,
        f"Full Body Health Checkup Package for Tk {s['amt']} (30% off) this month! Book appointment: {s['phone']}")
    add("5_health_checkup_pkg", "bangla", 5, "NONE", 0, False,
        f"ডায়াবেটিস ও হার্ট চেকআপ প্যাকেজে ২৫% ছাড়! আজই বুকিং দিন।")
    add("5_health_checkup_pkg", "banglish", 5, "NONE", 0, False,
        f"Senior citizen health checkup package Tk {s['amt']}. Call {s['phone']}.")
    add("5_health_checkup_pkg", "mixed", 5, "NONE", 0, False,
        f"Dental scaling & polishing 40% discount offer till Friday.")

    # 5_furniture_sale
    add("5_furniture_sale", "english", 5, "NONE", 0, False,
        f"Upgrade your home! Flat 20% discount on sofas, beds, and dining sets at our showroom.")
    add("5_furniture_sale", "bangla", 5, "NONE", 0, False,
        f"ঘর সাজান নতুন ফার্নিচারে! সব খাট ও সোফায় ২০% ছাড় ও ফ্রি ফিটিং।")
    add("5_furniture_sale", "banglish", 5, "NONE", 0, False,
        f"Home decor & furniture exhibition e 30% discount cholche!")
    add("5_furniture_sale", "mixed", 5, "NONE", 0, False,
        f"Office chair & desk combo offer Tk {s['amt']}. Visit Mirpur showroom.")

    return rows


def paraphrase_real_row(row: pd.Series, variant_idx: int, rng: random.Random) -> dict:
    """Create genuinely rephrased/restructured paraphrase variants of a real sample.csv row."""
    txt = str(row["text"]).strip()
    lbl = int(row["label"])
    tech = str(row["techniques"])
    act = int(row["action"])
    rid = int(row["id"])

    # Detect language roughly
    has_bn = bool(re.search(r"[\u0980-\u09FF]", txt))
    lang = "bangla" if has_bn else rng.choice(["english", "banglish", "mixed"])

    # Clean turns if multi-turn
    turns = [t.strip() for t in txt.split("|||")]
    is_mt = len(turns) > 1

    # Restructuring strategies based on variant_idx (0..9)
    if variant_idx % 5 == 0:
        # Reverse/front-load call-to-action or add contextual framing
        if is_mt:
            spk = rng.choice(["Agent", "Caller", "Sender"])
            other = rng.choice(["Customer", "You", "Receiver"])
            new_turns = [f"{spk}: Hello, ekta kotha chilo.", f"{other}: Ji bolun."]
            for i, t in enumerate(turns):
                clean_t = re.sub(r"^(S|R|A|B|Caller|You|Agent|Customer):\s*", "", t)
                new_turns.append(f"{spk if i % 2 == 0 else other}: {clean_t}")
            new_txt = " ||| ".join(new_turns)
            n_t = len(new_turns)
        else:
            prefix = rng.choice([
                "Notice: ", "Attention: ", "Update: ", "Info: ", "জরুরি বার্তা: ", "Shunun: "
            ]) if not has_bn else rng.choice(["জরুরি বার্তা: ", "তথ্য: ", "বিজ্ঞপ্তি: "])
            suffix = rng.choice([
                " Please check.", " Kind regards.", " Thanks.", " ধন্যবাদ।" if has_bn else " Dhonyobad."
            ])
            new_txt = prefix + txt + suffix
            n_t = 1
    elif variant_idx % 5 == 1:
        # Convert single-turn to 3-turn conversation (or expand existing multi-turn)
        if not is_mt:
            spk_s, spk_r = rng.choice([("S", "R"), ("Caller", "You"), ("A", "B")])
            new_txt = f"{spk_s}: Salam, free achen? ||| {spk_r}: Ji bolun. ||| {spk_s}: {txt}"
            n_t = 3
        else:
            # Drop speaker tags and rephrase into compact narrative
            joined = " ".join(re.sub(r"^(S|R|A|B|Caller|You|Agent|Customer):\s*", "", t) for t in turns)
            new_txt = f"Conversation summary: {joined}"
            n_t = 1
    elif variant_idx % 5 == 2:
        # Synonym substitution + sentence-level rephrasing
        swaps = [
            ("successful", "completed"), ("Successful", "Completed"),
            ("Do not share", "Never disclose"), ("never ask for", "do not request"),
            ("Click to claim", "Tap the link to collect"), ("click", "visit"),
            ("Pay", "Send"), ("pay", "transfer"),
            ("discount", "price off"), ("offer", "deal"), ("Offer", "Deal"),
            ("held", "stopped"), ("expires", "ends"), ("within 1 hour", "before 60 minutes"),
            ("bolen", "janan"), ("pathan", "send korun"), ("korun", "koren"),
            ("taka", "BDT"), ("ferot din", "return korun"), ("jitechen", "won"),
            ("সফল হয়েছে", "সম্পন্ন হয়েছে"), ("ক্লিক করুন", "ভিজিট করুন"),
            ("পাঠান", "শেয়ার করুন"), ("টাকা", "টাকা"),
        ]
        new_txt = txt
        for a, b in swaps:
            if a in new_txt and rng.random() < 0.7:
                new_txt = new_txt.replace(a, b, 1)
        if new_txt == txt:
            new_txt = f"Message received: {txt}"
        n_t = len(turns)
    elif variant_idx % 5 == 3:
        # Clause reordering if sentence has two parts
        parts = re.split(r"(?<=[.!।])\s+", txt)
        if len(parts) >= 2 and not is_mt:
            reordered = parts[-1] + " " + " ".join(parts[:-1])
            new_txt = reordered.strip()
        else:
            new_txt = f"FYI - {txt}"
        n_t = len(turns)
    else:
        # Conversational inquiry wrapper
        if has_bn:
            new_txt = f"S: ভাই একটা মেসেজ দেখলাম। ||| R: কী মেসেজ? ||| S: {txt}"
            n_t = 3
        else:
            new_txt = f"A: Hey, got this message. ||| B: What does it say? ||| A: {txt}"
            n_t = 3

    return {
        "text": new_txt,
        "label": lbl,
        "techniques": "|".join(sorted(tech.split("|"))) if tech != "NONE" else "NONE",
        "action": act,
        "family_id": f"real_para_{rid}",
        "scenario_id": f"real_para_{rid}",
        "language": lang,
        "n_turns": n_t,
        "is_hard_negative": lbl in (0, 1, 4, 5),
        "source": "real_paraphrase",
    }


def generate_synthetic_v2(target_rows: int = 15000, seed: int = 42) -> pd.DataFrame:
    rng = random.Random(seed)

    # 1. Generate 10 paraphrases per real row (860 rows)
    real_df = pd.read_csv("sample.csv")
    paraphrase_rows = []
    for _, row in real_df.iterrows():
        for v_idx in range(10):
            paraphrase_rows.append(paraphrase_real_row(row, v_idx, rng))

    # 2. Remaining target (~14,140) filled from v1 + expanded v2 scenarios
    rem_target = target_rows - len(paraphrase_rows)
    # Balanced across all 6 classes (with slight extra weight for 0 and 5)
    target_per_class = {
        0: int(rem_target * 0.18),
        1: int(rem_target * 0.16),
        2: int(rem_target * 0.16),
        3: int(rem_target * 0.16),
        4: int(rem_target * 0.16),
        5: rem_target - int(rem_target * 0.18) - 4 * int(rem_target * 0.16),
    }

    buckets = {k: [] for k in range(6)}

    while any(len(buckets[k]) < target_per_class[k] for k in range(6)):
        slots = _build_slots(rng)

        # Collect v1 templates
        for fam, lang, lbl, tech, act, hard, txt in _gen_single_templates(slots):
            if len(buckets[lbl]) < target_per_class[lbl] and rng.random() < 0.25:
                buckets[lbl].append({
                    "text": txt,
                    "label": lbl,
                    "techniques": "|".join(sorted(tech.split("|"))) if tech != "NONE" else "NONE",
                    "action": act,
                    "family_id": fam,
                    "scenario_id": V1_FAMILY_TO_SCENARIO[fam],
                    "language": lang,
                    "n_turns": 1,
                    "is_hard_negative": hard,
                    "source": "synthetic_v1",
                })

        for fam, lang, lbl, tech, act, hard, n_t, txt in _gen_multiturn_templates(rng, slots):
            if len(buckets[lbl]) < target_per_class[lbl] and rng.random() < 0.45:
                buckets[lbl].append({
                    "text": txt,
                    "label": lbl,
                    "techniques": "|".join(sorted(tech.split("|"))) if tech != "NONE" else "NONE",
                    "action": act,
                    "family_id": fam,
                    "scenario_id": V1_FAMILY_TO_SCENARIO[fam],
                    "language": lang,
                    "n_turns": n_t,
                    "is_hard_negative": hard,
                    "source": "synthetic_v1",
                })

        # Collect new v2 scenarios (both single and multi-turn wrapped variants to hit ~35% multi-turn)
        for scen, fam, lang, lbl, tech, act, n_t, hard, txt in _gen_new_scenarios(rng, slots):
            if len(buckets[lbl]) >= target_per_class[lbl]:
                continue
            # Also randomly wrap ~28% of single-turn new scenarios into multi-turn delayed reveal
            if n_t == 1 and rng.random() < 0.28:
                n_t, txt = _wrap_multiturn(rng, slots, [txt], rng.choice(["greet", "casual", "bangla"]))
            buckets[lbl].append({
                "text": txt,
                "label": lbl,
                "techniques": "|".join(sorted(tech.split("|"))) if tech != "NONE" else "NONE",
                "action": act,
                "family_id": fam,
                "scenario_id": scen,
                "language": lang,
                "n_turns": n_t,
                "is_hard_negative": hard,
                "source": "synthetic_v2",
            })

    all_rows = paraphrase_rows[:]
    for k in range(6):
        all_rows.extend(buckets[k][:target_per_class[k]])

    rng.shuffle(all_rows)
    df = pd.DataFrame(all_rows)

    # Apply noise to ~30% of rows
    is_noisy = []
    noisy_texts = []
    for txt in df["text"].tolist():
        if rng.random() < 0.30:
            noisy_texts.append(apply_noise(txt, rng, level=0.35))
            is_noisy.append(True)
        else:
            noisy_texts.append(txt)
            is_noisy.append(False)
    df["text"] = noisy_texts
    df["is_noisy"] = is_noisy

    df.insert(0, "id", range(200001, 200001 + len(df)))
    return df

In [5]:
df_v2 = generate_synthetic_v2(target_rows=15000, seed=SEED)
os.makedirs('data', exist_ok=True)
df_v2.to_csv('data/synthetic_v2.csv', index=False)
print('Generated data/synthetic_v2.csv:', df_v2.shape)


Generated data/synthetic_v2.csv: (15000, 12)


## 4. Validation, Phase 3A Checks, & `submission_phase3a.csv`

In [6]:
import os
import sys
import pandas as pd

REQUIRED_COLS = ["id", "label", "techniques", "action", "confidence"]
VALID_CODES = {"T1", "T2", "T3", "T4", "T5", "T6"}


def get_data_dir():
    return "data" if os.path.isfile(os.path.join("data", "sample.csv")) else "."


def check_techniques(val: str) -> bool:
    if not isinstance(val, str):
        return False
    if val != val.strip() or not val or " " in val:
        return False
    if val == "NONE":
        return True
    parts = val.split("|")
    if "NONE" in parts:
        return False
    if any(p not in VALID_CODES for p in parts):
        return False
    if len(parts) != len(set(parts)):
        return False
    if parts != sorted(parts):
        return False
    return True


def validate_submission(sub_path: str = "submission.csv") -> bool:
    data_dir = DATA_DIR
    ref_path = os.path.join(data_dir, "sample_submission.csv")
    test_path = os.path.join(data_dir, "test.csv")

    failures = []

    if not os.path.isfile(sub_path):
        print(f"FAIL: File not found: {sub_path}")
        return False

    sub = pd.read_csv(sub_path, keep_default_na=False)
    ref = pd.read_csv(ref_path, usecols=["id"])
    test = pd.read_csv(test_path, usecols=["id"])

    # 1. Header matches exactly
    if list(sub.columns) != REQUIRED_COLS:
        failures.append(f"Header mismatch: got {list(sub.columns)}, expected {REQUIRED_COLS}")

    # 2. Row count & ID set / order / duplicates
    if len(sub) != len(test):
        failures.append(f"Row count mismatch: got {len(sub)}, expected {len(test)}")
    if len(sub) != len(ref):
        failures.append(f"Row count mismatch vs sample_submission: got {len(sub)}, expected {len(ref)}")

    try:
        sub_ids = pd.to_numeric(sub["id"], errors="raise")
        if sub_ids.duplicated().any():
            failures.append(f"Duplicate IDs found: {sub_ids.duplicated().sum()}")
        if len(sub_ids) == len(test) and not (sub_ids.values == test["id"].values).all():
            failures.append("IDs do not match test.csv order or values")
    except Exception as e:
        failures.append(f"ID parsing error: {e}")

    # 3. No nulls or blank strings anywhere
    for col in REQUIRED_COLS:
        if col in sub.columns:
            blank_cnt = (sub[col].astype(str).str.strip() == "").sum()
            if blank_cnt > 0:
                failures.append(f"Column '{col}' has {blank_cnt} blank/null values")

    # 4. label and action are integers in 0-5
    for col in ["label", "action"]:
        if col in sub.columns:
            try:
                vals = pd.to_numeric(sub[col], errors="raise")
                if (vals % 1 != 0).any():
                    failures.append(f"Column '{col}' has non-integer values")
                if (~vals.isin([0, 1, 2, 3, 4, 5])).any():
                    failures.append(f"Column '{col}' has values outside 0-5")
            except Exception as e:
                failures.append(f"Column '{col}' parsing error: {e}")

    # 5. techniques validation
    if "techniques" in sub.columns:
        bad_tech = (~sub["techniques"].apply(check_techniques)).sum()
        if bad_tech > 0:
            failures.append(f"Column 'techniques' has {bad_tech} invalid entries")

    # 6. confidence numeric in [0, 1] and not NaN
    if "confidence" in sub.columns:
        try:
            confs = pd.to_numeric(sub["confidence"], errors="raise")
            if confs.isna().any() or ((confs < 0.0) | (confs > 1.0)).any():
                failures.append("Column 'confidence' has NaN or values outside [0, 1]")
        except Exception as e:
            failures.append(f"Column 'confidence' parsing error: {e}")

    if failures:
        print("FAIL:")
        for f in failures:
            print("  -", f)
        return False
    else:
        print("PASS")
        return True

import os
import random
import time
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import f1_score, confusion_matrix


SEED = 42
TECH_CODES = ["T1", "T2", "T3", "T4", "T5", "T6"]


def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)


def encode_techniques(series: pd.Series) -> np.ndarray:
    y = np.zeros((len(series), len(TECH_CODES)), dtype=int)
    for i, val in enumerate(series.astype(str)):
        parts = val.split("|")
        for j, code in enumerate(TECH_CODES):
            if code in parts:
                y[i, j] = 1
    return y


def postprocess_predictions(labels, tech_probs, actions, confs):
    out_techs = []
    out_actions = actions.copy()
    for i in range(len(labels)):
        if labels[i] == 0:
            out_techs.append("NONE")
            out_actions[i] = 0
        else:
            active = [TECH_CODES[j] for j in range(len(TECH_CODES)) if tech_probs[i, j] >= 0.5]
            if not active:
                active = [TECH_CODES[int(np.argmax(tech_probs[i]))]]
            out_techs.append("|".join(sorted(active)))
    out_confs = np.round(np.clip(confs, 0.05, 0.98), 4)
    return labels, out_techs, out_actions, out_confs


def fit_vectorizers(texts):
    char_vec = TfidfVectorizer(
        analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True, dtype=np.float32
    )
    word_vec = TfidfVectorizer(
        analyzer="word", ngram_range=(1, 2), sublinear_tf=True, dtype=np.float32
    )
    X_char = char_vec.fit_transform(texts)
    X_word = word_vec.fit_transform(texts)
    return char_vec, word_vec, hstack([X_char, X_word]).tocsr()


def transform_vectorizers(char_vec, word_vec, texts):
    return hstack([char_vec.transform(texts), word_vec.transform(texts)]).tocsr()


def fit_and_predict(tr_texts, tr_lbl, tr_tech, tr_act, va_texts, sample_weight=None):
    cv_c, cv_w, X_tr = fit_vectorizers(tr_texts)
    X_va = transform_vectorizers(cv_c, cv_w, va_texts)

    ca = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED)
    ca.fit(X_tr, tr_lbl, sample_weight=sample_weight)
    pa = ca.predict_proba(X_va)
    pred_lbl = np.argmax(pa, axis=1)
    pred_conf = np.max(pa, axis=1)

    cb = OneVsRestClassifier(
        LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED)
    )
    cb.fit(X_tr, tr_tech)
    prob_tech = cb.predict_proba(X_va)

    cc = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED)
    cc.fit(X_tr, tr_act, sample_weight=sample_weight)
    pred_act = cc.predict(X_va)

    return postprocess_predictions(pred_lbl, prob_tech, pred_act, pred_conf)


def compute_metrics(y_lbl, y_tech, y_act, p_lbl, p_tech_strs, p_act):
    p_tech_bin = encode_techniques(pd.Series(p_tech_strs))
    f1_a = f1_score(y_lbl, p_lbl, average="macro")
    f1_b = float(np.mean([f1_score(y_tech[:, j], p_tech_bin[:, j], zero_division=0) for j in range(6)]))
    f1_c = f1_score(y_act, p_act, average="macro")
    return f1_a, f1_b, f1_c

import os
import random
import time
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import normalize



def run_phase3a_check():
    random.seed(SEED)
    np.random.seed(SEED)
    rng = random.Random(SEED)
    t0 = time.time()

    real_df = pd.read_csv("sample.csv")
    v2_df = pd.read_csv("data/synthetic_v2.csv", keep_default_na=False)

    real_clean = real_df["text"].fillna("").astype(str).apply(preprocess_text).tolist()
    real_noisy = [preprocess_text(apply_noise(t, rng, level=0.35)) for t in real_df["text"].fillna("").astype(str)]
    real_lbl = real_df["label"].astype(int).values
    real_tech = encode_techniques(real_df["techniques"])
    real_act = real_df["action"].astype(int).values
    real_ids = real_df["id"].astype(int).values

    # ==================== 1A. REAL-ROW CHECK: synthetic_v2 ONLY ====================
    print("=" * 70)
    print("1A. REAL-ROW CHECK: TRAIN ON synthetic_v2 ONLY (NO real_paraphrase, NO synthetic_v1)")
    print("=" * 70)
    s2_only = v2_df[v2_df["source"] == "synthetic_v2"].reset_index(drop=True)
    s2_proc = s2_only["text"].astype(str).apply(preprocess_text).tolist()
    s2_lbl = s2_only["label"].astype(int).values
    s2_tech = encode_techniques(s2_only["techniques"])
    s2_act = s2_only["action"].astype(int).values
    print(f"Training rows (source=='synthetic_v2'): {len(s2_only)}")

    pl_c, pt_c, pa_c, _ = fit_and_predict(s2_proc, s2_lbl, s2_tech, s2_act, real_clean)
    pl_n, pt_n, pa_n, _ = fit_and_predict(s2_proc, s2_lbl, s2_tech, s2_act, real_noisy)

    fa_c, fb_c, fc_c = compute_metrics(real_lbl, real_tech, real_act, pl_c, pt_c, pa_c)
    fa_n, fb_n, fc_n = compute_metrics(real_lbl, real_tech, real_act, pl_n, pt_n, pa_n)
    print(f"CLEAN 86 Real: Task A={fa_c:.4f}, Task B={fb_c:.4f}, Task C={fc_c:.4f}")
    print("Confusion Matrix (CLEAN Task A):\n", confusion_matrix(real_lbl, pl_c))
    print(f"NOISY 86 Real: Task A={fa_n:.4f}, Task B={fb_n:.4f}, Task C={fc_n:.4f}")
    print("Confusion Matrix (NOISY Task A):\n", confusion_matrix(real_lbl, pl_n))

    # ==================== 1B. 5-FOLD OVER REAL ROWS WITH NON-LEAKING PARAPHRASES ====================
    print("\n" + "=" * 70)
    print("1B. 5-FOLD OVER REAL ROWS: synthetic_v2 + TRAIN-FOLD real_paraphrase ONLY")
    print("=" * 70)
    para_df = v2_df[v2_df["source"] == "real_paraphrase"].copy()
    para_df["parent_id"] = para_df["scenario_id"].str.replace("real_para_", "", regex=False).astype(int)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    oof_c_lbl, oof_c_tech, oof_c_act = np.zeros(len(real_df), int), [""] * len(real_df), np.zeros(len(real_df), int)
    oof_n_lbl, oof_n_tech, oof_n_act = np.zeros(len(real_df), int), [""] * len(real_df), np.zeros(len(real_df), int)

    for tr_i, va_i in skf.split(real_clean, real_lbl):
        tr_parent_ids = set(real_ids[tr_i])
        fold_para = para_df[para_df["parent_id"].isin(tr_parent_ids)]
        fp_proc = fold_para["text"].astype(str).apply(preprocess_text).tolist()
        fp_lbl = fold_para["label"].astype(int).values
        fp_tech = encode_techniques(fold_para["techniques"])
        fp_act = fold_para["action"].astype(int).values

        tr_texts = s2_proc + fp_proc
        tr_lbl = np.concatenate([s2_lbl, fp_lbl])
        tr_tech = np.vstack([s2_tech, fp_tech])
        tr_act = np.concatenate([s2_act, fp_act])

        plc, ptc, pac, _ = fit_and_predict(tr_texts, tr_lbl, tr_tech, tr_act, [real_clean[i] for i in va_i])
        pln, ptn, pan, _ = fit_and_predict(tr_texts, tr_lbl, tr_tech, tr_act, [real_noisy[i] for i in va_i])
        for k, idx in enumerate(va_i):
            oof_c_lbl[idx], oof_c_tech[idx], oof_c_act[idx] = plc[k], ptc[k], pac[k]
            oof_n_lbl[idx], oof_n_tech[idx], oof_n_act[idx] = pln[k], ptn[k], pan[k]

    fa_1b_c, fb_1b_c, fc_1b_c = compute_metrics(real_lbl, real_tech, real_act, oof_c_lbl, oof_c_tech, oof_c_act)
    fa_1b_n, fb_1b_n, fc_1b_n = compute_metrics(real_lbl, real_tech, real_act, oof_n_lbl, oof_n_tech, oof_n_act)
    print(f"CLEAN 86 Real (5-fold w/ train paraphrases): Task A={fa_1b_c:.4f}, Task B={fb_1b_c:.4f}, Task C={fc_1b_c:.4f}")
    print("Confusion Matrix (CLEAN Task A):\n", confusion_matrix(real_lbl, oof_c_lbl))
    print(f"NOISY 86 Real (5-fold w/ train paraphrases): Task A={fa_1b_n:.4f}, Task B={fb_1b_n:.4f}, Task C={fc_1b_n:.4f}")
    print("Confusion Matrix (NOISY Task A):\n", confusion_matrix(real_lbl, oof_n_lbl))

    # ==================== 2. CROSS-SCENARIO LEAKAGE ====================
    print("\n" + "=" * 70)
    print("2. CROSS-SCENARIO LEAKAGE (NEAREST NEIGHBOR FROM DIFFERENT scenario_id)")
    print("=" * 70)
    # Check across all synthetic rows (v1 + v2)
    cs_df = v2_df[v2_df["source"] != "real_paraphrase"].reset_index(drop=True)
    cs_proc = cs_df["text"].astype(str).apply(preprocess_text).tolist()
    cs_scens = cs_df["scenario_id"].values
    cs_lbls = cs_df["label"].astype(int).values

    char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True, dtype=np.float32)
    X_cs = normalize(char_vec.fit_transform(cs_proc))

    n_cs = len(cs_df)
    nn_sims = np.zeros(n_cs, dtype=np.float32)
    nn_idx = np.zeros(n_cs, dtype=int)

    # Batched cosine similarity to stay fast & low memory
    batch_sz = 1000
    for start in range(0, n_cs, batch_sz):
        end = min(start + batch_sz, n_cs)
        sims = (X_cs[start:end] @ X_cs.T).toarray()
        # Mask out same scenario_id
        for r in range(end - start):
            sims[r, cs_scens == cs_scens[start + r]] = -1.0
        best_j = np.argmax(sims, axis=1)
        nn_idx[start:end] = best_j
        nn_sims[start:end] = sims[np.arange(end - start), best_j]

    same_lbl_pct = (cs_lbls[nn_idx] == cs_lbls).mean() * 100
    print(f"Rows evaluated (synthetic_v1 + synthetic_v2): {n_cs}")
    print(f"Cross-scenario max similarity:")
    print(f"  Mean   : {nn_sims.mean():.4f}")
    print(f"  Median : {np.median(nn_sims):.4f}")
    print(f"  >= 0.8 : {(nn_sims >= 0.8).mean() * 100:.2f}% ({(nn_sims >= 0.8).sum()}/{n_cs})")
    print(f"  Nearest neighbor has SAME label: {same_lbl_pct:.2f}%")

    # Top 10 closest cross-scenario pairs with DIFFERENT labels
    diff_mask = cs_lbls[nn_idx] != cs_lbls
    diff_indices = np.where(diff_mask)[0]
    top_diff = diff_indices[np.argsort(-nn_sims[diff_indices])]
    print("\nTop 10 closest cross-scenario pairs with DIFFERENT labels:")
    seen_pairs = set()
    printed = 0
    for i in top_diff:
        j = nn_idx[i]
        pair_key = tuple(sorted([cs_scens[i], cs_scens[j]]))
        if pair_key in seen_pairs:
            continue
        seen_pairs.add(pair_key)
        printed += 1
        print(f"  [{printed:02d}] sim={nn_sims[i]:.4f} | L{cs_lbls[i]}({cs_scens[i]}) <-> L{cs_lbls[j]}({cs_scens[j]})")
        print(f"       A: {repr(cs_df.loc[i, 'text'][:110])}")
        print(f"       B: {repr(cs_df.loc[j, 'text'][:110])}")
        if printed >= 10:
            break

    # ==================== 3. FINAL PHASE 3A MODEL & PREDICT test.csv ====================
    print("\n" + "=" * 70)
    print("3. TRAIN ON sample.csv (5x) + ALL synthetic_v2 -> submission_phase3a.csv")
    print("=" * 70)
    v2_proc = v2_df["text"].astype(str).apply(preprocess_text).tolist()
    v2_lbl = v2_df["label"].astype(int).values
    v2_tech = encode_techniques(v2_df["techniques"])
    v2_act = v2_df["action"].astype(int).values

    full_texts = real_clean + v2_proc
    full_lbl = np.concatenate([real_lbl, v2_lbl])
    full_tech = np.vstack([real_tech, v2_tech])
    full_act = np.concatenate([real_act, v2_act])
    full_weights = np.concatenate([np.full(len(real_lbl), 5.0), np.ones(len(v2_lbl))])

    char_v, word_v, X_full = fit_vectorizers(full_texts)
    clf_a = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED).fit(X_full, full_lbl, sample_weight=full_weights)
    clf_b = OneVsRestClassifier(LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED)).fit(X_full, full_tech)
    clf_c = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED).fit(X_full, full_act, sample_weight=full_weights)

    out_path = "submission_phase3a.csv"
    first_chunk = True
    label_counts = {k: 0 for k in range(6)}
    total_rows = 0

    for chunk in pd.read_csv("test.csv", chunksize=50000):
        clean_batch = chunk["text"].fillna("").astype(str).apply(preprocess_text).tolist()
        X_test = transform_vectorizers(char_v, word_v, clean_batch)
        probs_a = clf_a.predict_proba(X_test)
        preds_a = np.argmax(probs_a, axis=1)
        confs = np.max(probs_a, axis=1)
        tech_probs = clf_b.predict_proba(X_test)
        preds_c = clf_c.predict(X_test)

        preds_a, tech_strs, preds_c, confs = postprocess_predictions(preds_a, tech_probs, preds_c, confs)
        for lbl in preds_a:
            label_counts[int(lbl)] += 1

        pd.DataFrame({
            "id": chunk["id"].values.astype(int),
            "label": preds_a.astype(int),
            "techniques": tech_strs,
            "action": preds_c.astype(int),
            "confidence": confs,
        }).to_csv(out_path, mode="w" if first_chunk else "a", header=first_chunk, index=False)
        first_chunk = False
        total_rows += len(chunk)

    print(f"Saved {out_path} ({total_rows:,} rows) in {time.time() - t0:.2f}s")
    print("Phase 3A Predicted Label Distribution on test.csv:")
    for k, v in label_counts.items():
        print(f"  Label {k}: {v:7,d} ({v/total_rows*100:5.2f}%)")

    print("\nValidating submission_phase3a.csv:")
    validate_submission(out_path)


if __name__ == "__main__":
    run_phase3a_check()

1A. REAL-ROW CHECK: TRAIN ON synthetic_v2 ONLY (NO real_paraphrase, NO synthetic_v1)


Training rows (source=='synthetic_v2'): 13168


CLEAN 86 Real: Task A=0.8370, Task B=0.7085, Task C=0.8145
Confusion Matrix (CLEAN Task A):
 [[18  7  1  0  3  3]
 [ 0 11  1  0  0  0]
 [ 0  0 10  0  0  0]
 [ 0  0  0 11  0  0]
 [ 0  0  0  0  9  0]
 [ 0  0  0  1  0 11]]
NOISY 86 Real: Task A=0.8034, Task B=0.6958, Task C=0.8396
Confusion Matrix (NOISY Task A):
 [[19  6  1  0  3  3]
 [ 0 10  2  0  0  0]
 [ 0  0  9  0  0  1]
 [ 0  0  0 11  0  0]
 [ 0  0  0  0  9  0]
 [ 0  0  0  2  0 10]]

1B. 5-FOLD OVER REAL ROWS: synthetic_v2 + TRAIN-FOLD real_paraphrase ONLY


CLEAN 86 Real (5-fold w/ train paraphrases): Task A=0.8771, Task B=0.7144, Task C=0.8428
Confusion Matrix (CLEAN Task A):
 [[24  3  1  0  2  2]
 [ 0 11  1  0  0  0]
 [ 0  0 10  0  0  0]
 [ 0  0  0 11  0  0]
 [ 0  0  0  0  9  0]
 [ 0  0  0  2  0 10]]
NOISY 86 Real (5-fold w/ train paraphrases): Task A=0.7771, Task B=0.6838, Task C=0.8106
Confusion Matrix (NOISY Task A):
 [[19  5  1  0  3  4]
 [ 0 10  2  0  0  0]
 [ 1  0  8  0  0  1]
 [ 0  0  0 11  0  0]
 [ 0  0  0  0  9  0]
 [ 0  0  0  3  0  9]]

2. CROSS-SCENARIO LEAKAGE (NEAREST NEIGHBOR FROM DIFFERENT scenario_id)


Rows evaluated (synthetic_v1 + synthetic_v2): 14140
Cross-scenario max similarity:
  Mean   : 0.3470
  Median : 0.3411
  >= 0.8 : 0.15% (21/14140)
  Nearest neighbor has SAME label: 61.42%

Top 10 closest cross-scenario pairs with DIFFERENT labels:
  [01] sim=0.7362 | L2(2_eid_gift_scam) <-> L4(4_damaged_parcel_comp)
       A: 'Agent: Assalamu alaikum Mitu, kemon achen? ||| Customer: Wa alaikumussalam, bhalo. Ke bolchen? ||| Agent: Eid '
       B: 'Agent: Assalamu alaikum Mitu, kemon achen? ||| Customer: Wa alaikumussalam, bhalo. Ke bolchen? ||| Agent: Clai'
  [02] sim=0.6975 | L2(2_telco_bundle_gift) <-> L5(5_telco_bundles)
       A: 'Agent: Assalamu alaikum Nusrat, kemon achen? ||| Customer: Wa alaikumussalam, bhalo. Ke bolchen? ||| Agent: Ap'
       B: 'Agent: Assalamu alaikum Nusrat, kemon achen? ||| Customer: Wa alaikumussalam, bhalo. Ke bolchen? ||| Agent: Ro'
  [03] sim=0.6956 | L1(1_pwd_reset_link) <-> L4(4_courier_pin_confirm)
       A: 'Caller: Assalamu alaikum Rahim, kemon a

Saved submission_phase3a.csv (500,000 rows) in 235.52s
Phase 3A Predicted Label Distribution on test.csv:
  Label 0: 156,366 (31.27%)
  Label 1:  92,314 (18.46%)
  Label 2:  51,580 (10.32%)
  Label 3:  48,105 ( 9.62%)
  Label 4:  65,301 (13.06%)
  Label 5:  86,334 (17.27%)

Validating submission_phase3a.csv:


PASS
